# 10장. 분류 분석으로 주문 취소 여부 예측하기

이 노트북은 완료 주문과 취소 주문만 사용해 `is_cancelled`를 예측하는 이진 분류 실습입니다.

## 학습 목표

- `completed=0`, `cancelled=1`로 타깃 범위를 명확하게 정의합니다.
- `refunded`와 기타 상태를 이진 분류 대상에서 제외합니다.
- 병합의 키 관계, 행 수, 미매칭을 검증합니다.
- train, validation, test를 분리합니다.
- Dummy 기준 모델, Logistic Regression, Random Forest를 비교합니다.
- validation에서 모델과 임계값을 선택하고 test는 최종 평가에 한 번만 사용합니다.
- 예측 결과에서 고객명 등 불필요한 개인정보를 제외합니다.


## 제출 정보

- 이름: 유은송
- GitHub ID: song-03
- 작성일: 2026.11.15. 
- 최종 Notebook URL: https://github.com/song-03/llm-data-analysis-study/blob/main/chapter10/chapter10.ipynb


## 1. 프로젝트 루트와 실행 환경 설정

전처리 파일이 없다면 프로젝트 루트에서 먼저 다음 명령을 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
프로젝트 루트: C:\dev\llm-data-analysis-course
전처리 데이터 폴더: C:\dev\llm-data-analysis-course\data\processed
보고서 폴더: C:\dev\llm-data-analysis-course\reports


In [2]:
# CH10 지정 입력 준비: 기존 데이터의 시간 순서 오류를 숨기지 않고 준비 절차로 재검증
import subprocess
prepared = subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts' / 'prepare_ch10_data.py')], cwd=PROJECT_ROOT, capture_output=True, text=True, encoding='utf-8', errors='replace')
print(prepared.stdout)
print(prepared.stderr)
prepared.check_returncode()

Chapter10 모델링 입력 준비 완료
입력 Raw 경로: C:\dev\llm-data-analysis-course\data\raw

[관계 검증]
                                               check  invalid_count
  orders.customer_id exists in customers.customer_id              0
      order_items.order_id exists in orders.order_id              0
order_items.product_id exists in products.product_id              0

[Chapter10 입력 범위: 날짜를 변경하지 않고 시간 관계 오류 격리]
order_status  original_count  quarantined_count  prepared_count                                    policy
   cancelled              64                 10              54 exclude_signup_after_order_no_date_repair
   completed             184                 35             149 exclude_signup_after_order_no_date_repair
    refunded              52                 11              41 exclude_signup_after_order_no_date_repair

[저장된 processed 파일]
- C:\dev\llm-data-analysis-course\data\processed\customers_clean.csv
- C:\dev\llm-data-analysis-course\data\processed\products_clean.csv
- C:\dev\llm-data-a

## 2. 전처리 데이터 확인과 불러오기

이 장에서는 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.


In [3]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())


customers: (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders: (300, 7) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items: (764, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


## 3. 주문 상태와 타깃 범위 확인

이번 실습은 완료 주문과 취소 주문만 비교합니다.

- `completed` → 0
- `cancelled` → 1
- `refunded`와 기타 상태 → 학습 대상에서 제외


In [4]:
status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary


,order_status,order_count
0,completed,184
1,cancelled,64
2,refunded,52


## 4. 분류 데이터와 병합 검증표 생성

공통 함수는 주문 특징 집계, 안전한 병합, 시간 순서 점검, 타깃 생성까지 수행합니다.


In [5]:
# 시간 순서 오류: 원본 날짜를 수정하지 않고 유효 표본을 별도로 정의
source_orders = orders.copy()
source_items = order_items.copy()
time_audit = orders[['order_id', 'customer_id', 'order_status', 'order_date']].merge(
    customers[['customer_id', 'signup_date']], on='customer_id', how='left', validate='many_to_one', indicator=True)
order_dt = pd.to_datetime(time_audit['order_date'], errors='coerce')
signup_dt = pd.to_datetime(time_audit['signup_date'], errors='coerce')
assert time_audit['_merge'].eq('both').all()
assert order_dt.notna().all() and signup_dt.notna().all()
time_audit['signup_after_order'] = signup_dt.gt(order_dt)
scope_summary = pd.crosstab(time_audit['order_status'], time_audit['signup_after_order']).reindex(columns=[False, True], fill_value=0)
scope_summary.columns = ['valid_time_count', 'invalid_time_count']
scope_summary['original_count'] = scope_summary.sum(axis=1)
print('시간 순서별 원본 상태 분포'); display(scope_summary)
invalid_order_ids = set(time_audit.loc[time_audit['signup_after_order'], 'order_id'])
orders_for_model = orders.loc[~orders['order_id'].isin(invalid_order_ids)].copy()
order_items_for_model = order_items.loc[order_items['order_id'].isin(orders_for_model['order_id'])].copy()
print('주문 수:', len(orders), '->', len(orders_for_model), '변화:', len(orders_for_model)-len(orders))
print('상세 행 수:', len(order_items), '->', len(order_items_for_model), '변화:', len(order_items_for_model)-len(order_items))
print('날짜 오류 원인: 가입일 > 주문일 관계 확인. 어느 원본 날짜가 잘못됐는지는 미확인; 원본 파일 수정 없음.')
scope_summary.to_csv(REPORT_DIR / 'ch10_input_scope_checks.csv', encoding='utf-8-sig')

# 수정한 스크립트가 만든 분석용 입력과 Notebook 표본을 대조
CH10_PROCESSED_DIR = PROCESSED_DIR / 'ch10'
prepared_orders = pd.read_csv(CH10_PROCESSED_DIR / 'orders_clean.csv')
prepared_items = pd.read_csv(CH10_PROCESSED_DIR / 'order_items_clean.csv')
prepared_quarantine = pd.read_csv(CH10_PROCESSED_DIR / 'temporal_quarantine_internal.csv')
prepared_scope = pd.read_csv(CH10_PROCESSED_DIR / 'input_scope_checks.csv')
pd.testing.assert_frame_equal(orders_for_model.reset_index(drop=True),prepared_orders.reset_index(drop=True),check_dtype=False)
pd.testing.assert_frame_equal(order_items_for_model.reset_index(drop=True),prepared_items.reset_index(drop=True),check_dtype=False)
assert set(prepared_quarantine['order_id'])==invalid_order_ids
assert int(prepared_scope['quarantined_count'].sum())==len(invalid_order_ids)
quarantine_contract_ok=True
preparation_checks=pd.DataFrame([
 {'check':'notebook_prepared_orders_equal','value':len(prepared_orders),'status':'PASS'},
 {'check':'notebook_prepared_items_equal','value':len(prepared_items),'status':'PASS'},
 {'check':'quarantine_ids_equal_source_violations','value':len(prepared_quarantine),'status':'PASS'},
 {'check':'quarantined_binary_orders','value':int(prepared_quarantine['order_status'].isin(['completed','cancelled']).sum()),'status':'PASS'}])
preparation_checks.to_csv(REPORT_DIR/'ch10_input_preparation_checks.csv',index=False,encoding='utf-8-sig')
print('수정한 스크립트의 분석용 입력과 Notebook 표본 대조');display(preparation_checks)
print('원본 날짜는 미정정, 격리 정책으로 처리. 검증된 준비 입력만 모델링에 사용.')


시간 순서별 원본 상태 분포


,valid_time_count,invalid_time_count,original_count
order_status,,,
cancelled,54,10,64
completed,149,35,184
refunded,41,11,52


주문 수: 300 -> 244 변화: -56
상세 행 수: 764 -> 624 변화: -140
날짜 오류 원인: 가입일 > 주문일 관계 확인. 어느 원본 날짜가 잘못됐는지는 미확인; 원본 파일 수정 없음.
수정한 스크립트의 분석용 입력과 Notebook 표본 대조


,check,value,status
0,notebook_prepared_orders_equal,244,PASS
1,notebook_prepared_items_equal,624,PASS
2,quarantine_ids_equal_source_violations,56,PASS
3,quarantined_binary_orders,45,PASS


원본 날짜는 미정정, 격리 정책으로 처리. 검증된 준비 입력만 모델링에 사용.


In [6]:
from src.classification import (
    build_classification_dataset,
    target_distribution,
)

(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders_for_model,
    order_items=order_items_for_model,
)

target_dist = target_distribution(model_data)

print('모델링 데이터:', model_data.shape)
print('숫자형 feature:', numeric_features)
print('범주형 feature:', categorical_features)
display(target_dist)
display(merge_checks)
display(data_quality_checks)


모델링 데이터: (203, 16)
숫자형 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup']
범주형 feature: ['gender', 'city', 'payment_method']


,is_cancelled,class_label,count,ratio
0,0,completed,149,0.734
1,1,cancelled,54,0.266


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status
0,order_id → order_item_features,203,203,True,0,PASS
1,customer_id → customers,203,203,True,0,PASS


,check_item,count
0,binary_target_rows,203
1,completed_rows,149
2,cancelled_rows,54
3,excluded_status_rows,41
4,line_total_mismatch,0
5,negative_days_since_signup,0
6,missing_order_item_features,0
7,missing_customer_match,0
8,status_scope:completed,149
9,status_scope:cancelled,54


## 5. 데이터 누수 확인

`order_status`와 `is_cancelled`는 입력 feature에 포함되면 안 됩니다.


In [7]:
features = numeric_features + categorical_features
leakage_columns = {'order_status', 'is_cancelled'}
leakage_found = sorted(leakage_columns.intersection(features))

if leakage_found:
    raise ValueError(f'데이터 누수 위험 컬럼이 포함되었습니다: {leakage_found}')

print('사용 feature:', features)
print('데이터 누수 컬럼 없음')


사용 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup', 'gender', 'city', 'payment_method']
데이터 누수 컬럼 없음


In [8]:
# 원본/분석 표본을 구분한 품질·집계·Feature Evidence
import numpy as np
from pathlib import Path
checks=[]
def quality(name,count,scope='analysis',status=None):
    checks.append({'check_item':name,'count':int(count),'scope':scope,'status':status or ('PASS' if count==0 else 'FAIL')})
for name, frame, key in [('customers',customers,'customer_id'),('orders',source_orders,'order_id'),('order_items',source_items,'order_item_id')]:
    quality(name+'_duplicate_key',frame[key].duplicated().sum(),'source')
    quality(name+'_missing_key',frame[key].isna().sum(),'source')
quality('quantity_nonpositive',source_items['quantity'].le(0).sum(),'source')
quality('unit_price_nonpositive',source_items['unit_price'].le(0).sum(),'source')
quality('item_required_missing',source_items[['order_id','quantity','unit_price','line_total']].isna().sum().sum(),'source')
quality('source_line_total_mismatch',(~np.isclose(source_items['line_total'],source_items['quantity']*source_items['unit_price'],rtol=0,atol=1e-8)).sum(),'source')
quality('source_signup_after_order',time_audit['signup_after_order'].sum(),'source','QUARANTINED_UNCORRECTED')
item_features = order_items_for_model.groupby('order_id').agg(item_count=('order_item_id','size'), total_quantity=('quantity','sum'),order_amount=('line_total','sum')).reset_index()
quality('order_feature_duplicate',item_features['order_id'].duplicated().sum())
joined=model_data[['order_id','item_count','total_quantity','order_amount']].merge(item_features,on='order_id',how='left',validate='one_to_one',suffixes=('_actual','_expected'),indicator=True)
quality('aggregate_unmatched',joined['_merge'].ne('both').sum())
for col in ['item_count','total_quantity','order_amount']:
    quality(col+'_aggregate_mismatch',(~np.isclose(joined[col+'_actual'],joined[col+'_expected'],rtol=0,atol=1e-8)).sum())
quality('required_feature_missing',model_data[features].isna().sum().sum())
quality('analysis_negative_days_since_signup',model_data['days_since_signup'].lt(0).sum())
forbidden={'order_status','is_cancelled','order_id','customer_id','product_id','cancel_reason','cancelled_at','source_index','name','email','address'}
quality('forbidden_feature_overlap',len(forbidden.intersection(features)))
assert all(row['count']==0 for row in checks if row['status']!='QUARANTINED_UNCORRECTED')
feature_audit=pd.DataFrame([{'feature':f,'type':'numeric' if f in numeric_features else 'categorical','used':True,'forbidden':False,'availability':'주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인'} for f in features])
feature_audit['availability']='주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인'
for f in sorted(forbidden):
    feature_audit.loc[len(feature_audit)]={'feature':f,'type':'forbidden','used':False,'forbidden':True,'availability':'모델 입력 제외'}
extra_quality=pd.DataFrame(checks)
quality_evidence=pd.concat([data_quality_checks.assign(scope='analysis',status='INFO'),extra_quality],ignore_index=True)
for name,frame in [('ch10_target_distribution.csv',target_dist),('ch10_feature_audit.csv',feature_audit),('ch10_merge_checks.csv',merge_checks),('ch10_data_quality_checks.csv',quality_evidence)]:
    frame.to_csv(REPORT_DIR/name,index=False,encoding='utf-8-sig')
print('확장 품질 검증');display(extra_quality)
print('Feature Contract');display(feature_audit)
print('분석 대상 집계 상세 행:',len(order_items_for_model[order_items_for_model['order_id'].isin(model_data['order_id'])]))
print('일치 검증 허용오차: 절대오차 1e-8, 상대오차 0 (추가 검증 기준)')


확장 품질 검증


,check_item,count,scope,status
0,customers_duplicate_key,0,source,PASS
1,customers_missing_key,0,source,PASS
2,orders_duplicate_key,0,source,PASS
3,orders_missing_key,0,source,PASS
4,order_items_duplicate_key,0,source,PASS
5,order_items_missing_key,0,source,PASS
6,quantity_nonpositive,0,source,PASS
7,unit_price_nonpositive,0,source,PASS
8,item_required_missing,0,source,PASS
9,source_line_total_mismatch,0,source,PASS


Feature Contract


,feature,type,used,forbidden,availability
0,age,numeric,True,False,주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인
1,item_count,numeric,True,False,주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인
2,total_quantity,numeric,True,False,주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인
3,order_amount,numeric,True,False,주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인
4,order_month,numeric,True,False,주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인
5,order_dayofweek,numeric,True,False,주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인
6,days_since_signup,numeric,True,False,주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인
7,gender,categorical,True,False,주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인
8,city,categorical,True,False,주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인
9,payment_method,categorical,True,False,주문 생성 시점 이용 가능이라는 교육용 가정; 과거 시점 스냅샷은 미확인


분석 대상 집계 상세 행: 521
일치 검증 허용오차: 절대오차 1e-8, 상대오차 0 (추가 검증 기준)


## 1. 분류 문제 정의
- 타깃: `is_cancelled` (주문 1건의 취소 여부)
- 0의 의미: completed (완료 주문)
- 1의 의미: cancelled (취소 주문)
- 제외한 상태: refunded와 기타 상태 (refunded 52건, 기타 상태 0건)
- 예측 단위: 주문 1건
- 예측 시점: 주문 생성 직후 (교육용 가정)
- 예측 시점에 사용할 수 있는 정보: 주문 당시의 상품, 수량, 단가, 결제수단, 주문일, 고객 정보, 가입일 (사용할 수 있다고 가정)
- 예측 이후에만 알 수 있어 제외한 정보: 최종 주문 상태, 취소 사유, 취소 시각

### 결과 관찰

이번 실습에서는 주문 한 건이 취소될지를 예측했다. 타깃은 `is_cancelled`이고 completed를 0, cancelled를 1로 두었다. refunded는 완료와 성격이 다른 상태로 보고 기타 상태와 함께 분석에서 뺐다.

불러온 전처리 데이터에는 고객 150행, 주문 300행, 주문 상세 764행이 있었다. 주문은 완료 184건, 취소 64건, 환불 52건이었으며 기타 상태는 0건이었다. 날짜를 점검하자 가입일이 주문일보다 늦은 주문이 56건 나왔고 이 가운데 완료/취소 주문이 45건이었다.

| 상태 | 전체 | 날짜 오류로 제외 | 분석에 사용 |
| --- | ---: | ---: | ---: |
| completed | 184 | 35 | 149 |
| cancelled | 64 | 10 | 54 |
| refunded | 52 | 11 | 0 (타깃 범위에서 제외) |

날짜가 잘못된 주문 56건을 먼저 빼자 300건이 244건으로 줄었다. 여기서 남은 환불 주문 41건을 마저 제외해 최종 203건을 분석에 썼다. 근거 자료는 `ch10_target_distribution.csv`와 `ch10_data_quality_checks.csv`이다.

### 나의 해석과 판단

예측 시점은 주문 생성 직후로 정하고 그 시점에 주문 당시의 상품, 수량, 단가, 결제수단과 주문일, 고객 정보, 가입일을 쓸 수 있다고 가정했다. 최종 주문 상태나 취소 사유와 취소 시각은 그때 알 수 없는 정보라 입력에서 뺐다. 주문이 취소되기 전에 위험을 예상한다는 실습 목적에 맞추려면 이 선을 지켜야 한다고 봤다.

처음에는 공식 Notebook대로 전처리 데이터 300건을 공통 함수에 넣어 돌렸다. 실행은 가입일이 주문일보다 늦은 주문이 있다는 `ValueError`와 함께 멈췄고 완료/취소 주문 중 45건이 여기에 걸렸다. `scripts/run_classification_analysis.py`는 프로젝트 루트에서 실행하면 그보다 먼저 `src`를 찾지 못하는 import 오류가 났다. 두 문제를 풀려고 내 로컬에서 `scripts/prepare_ch10_data.py`에 날짜 오류 주문을 따로 빼는 단계를 넣고 `run_classification_analysis.py`는 경로를 등록한 뒤 이렇게 만든 `data/processed/ch10/` 입력을 읽도록 고쳤다. `src/classification.py`는 손대지 않았다. 두 스크립트의 수정본은 강사 저장소에 없다. 그래서 원본 저장소에서 이 Notebook을 그대로 실행하면 같은 결과가 나오지 않는다. 수정본은 `chapter10/scripts/`에 함께 올렸다.

날짜 오류는 가입일과 주문일 중 어느 쪽이 틀렸는지 알 수 없어 임의로 고치지 않고 해당 주문을 따로 보관한 채 분석에서 뺐다. 원래 거래 기록과 맞춰 날짜를 바로잡은 뒤 완료/취소 248건을 다시 분석하는 길도 있다. 다만 지금 가진 자료만으로 날짜를 정하는 것은 적절하지 않다고 판단했다.

### 업무·분석적 의미

이번 모델은 주문이 들어온 직후 취소 위험을 짚어 내는 데 쓸 만한지 살펴보려고 만들었다. 취소가 끝난 뒤에야 알게 되는 정보를 넣으면 실제 상황에서는 쓸 수 없는 결과가 나온다. 예측 시점을 먼저 정한 것도 그래서다.

### 한계와 추가 확인 사항

이번 결과는 날짜 조건을 만족한 203건에서 나왔다. 제외한 주문의 특징이 나머지와 다르다면 분석 결과도 영향을 받을 수 있다. 고객 정보와 주문 상세가 실제 주문 당시의 값인지, 나중에 수정된 값인지도 확인하지 못했다. 날짜 오류 56건은 원본에 남겨 두었다. 분석용 입력에서만 뺐으므로 원본 날짜는 여전히 바로잡히지 않은 상태다.


## 2. Feature Contract와 Leakage Audit

### 사용한 숫자형 Feature

`age`, `item_count`, `total_quantity`, `order_amount`, `order_month`, `order_dayofweek`, `days_since_signup` (7개)

### 사용한 범주형 Feature

`gender`, `city`, `payment_method` (3개)

### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
취소 이후 정보
```

`order_status`, `is_cancelled`, `order_id`, `customer_id`, `product_id`, `cancel_reason`, `cancelled_at`, `source_index`, `name`, `email`, `address`를 사용 금지 목록에 올려 점검했다. 이 11개 가운데 실제 입력에 들어간 변수는 0개, 입력값 결측도 0개였다. 현재 파일에 없는 컬럼도 금지 목록에 넣어 두었다. 사용 변수와 제외 변수는 `ch10_feature_audit.csv`에 정리했다.

### 결과 관찰

모델에는 숫자형 7개와 범주형 3개, 모두 10개 변수를 넣었다.

| 유형 | 사용 Feature | 주문 생성 직후에 알 수 있다고 본 이유 |
| --- | --- | --- |
| 숫자형 | `age` | 주문 당시 고객 나이 정보라는 교육용 가정 |
| 숫자형 | `item_count` | 생성 시 주문 상세 행 수 |
| 숫자형 | `total_quantity` | 생성 시 상세 수량 합계 |
| 숫자형 | `order_amount` | 검증된 생성 시 상세 금액 합계 |
| 숫자형 | `order_month`, `order_dayofweek` | 주문 생성일의 월과 요일 |
| 숫자형 | `days_since_signup` | 주문일과 가입일 간 경과일; 음수 주문 제외 |
| 범주형 | `gender`, `city` | 주문 시점 고객 정보라는 교육용 가정 |
| 범주형 | `payment_method` | 주문 생성 시 선택한 결제수단 |

숫자형은 중앙값으로 결측을 채우고 StandardScaler를 적용했다. 범주형은 최빈값으로 채운 다음 OneHotEncoder를 거쳤다. 학습 때 없던 범주를 처리하려고 `handle_unknown="ignore"`를 지정했다. 세 모델 모두 scaler가 학습한 데이터는 121행으로 Train 크기와 같았다. 평균과 숫자형 결측 대체값도 Train에서 따로 계산한 값과 일치했다. (전처리 학습 결과는 뒤의 `기준 모델과 학습 모델 비교` 단계에서 확인)

### 나의 해석과 판단

파일에 있는 컬럼이라도 예측에 쓸 수 있는 것과 없는 것이 있다고 봤다. 최종 주문 상태가 그런 예다. 현재 파일에는 있지만 주문 생성 직후에는 알 수 없는 값이라, 입력에 넣으면 정답을 미리 알려주는 셈이 된다. 주문 ID와 고객 ID는 데이터를 합칠 때만 쓰고 모델 입력에서는 뺐다.

주문 상세를 집계한 수량과 금액은 주문 생성 시점에 알 수 있는 값이라고 가정하고 썼다. LLM 검토에서도 입력 목록에 금지 변수가 없다는 의견이 나왔다. 코드에서도 막혀 있는지 보라는 지적이 있어 금지 목록과 입력 변수의 교집합이 0개임을 한 번 더 확인했다. 전체 데이터로 전처리를 먼저 학습시키는 방법은 검증 데이터와 평가 데이터의 정보가 섞일 수 있어 쓰지 않았다.

### 업무·분석적 의미

취소 위험 점수를 업무에 연결할 수 있으려면 실제 예측에 쓸 수 있는 변수만 골라야 한다. 금지 목록 11개 중 실제 입력에 들어간 변수는 0개였다. 결측이 0개였는데도 전처리와 모델을 하나의 Pipeline으로 묶어 새로운 데이터가 같은 순서로 처리되게 했다. `order_status`가 들어갔다면 성능은 거의 완벽하게 나왔겠지만 주문 생성 직후에는 알 수 없는 값이라 실제로는 쓸 수 없는 모델이 된다.

### 한계와 추가 확인 사항

age, city 같은 고객 정보가 주문 당시의 값인지는 확인이 더 필요하다. 주문 수량이나 금액도 이후 수정됐을 가능성을 확인해야 한다. gender와 city를 썼을 때 고객 집단별로 오류가 다르게 나타나는지는 별도로 살펴봐야 한다. 금지 변수 검사는 입력에 금지 변수가 있는지만 가려낼 뿐이고 이런 문제는 그 검사 범위 밖에 있다.

---

## 3. 주문 단위 특징과 병합 검증

### 계산 관계 확인

```text
line_total = quantity × unit_price
```

- 계산 불일치 건수: 0건 (불러온 주문 상세 764행 기준)
- 주문 특징 `order_id` 중복 건수: 0건

### 병합 검증

| 병합 | 관계 | 병합 전 행 수 | 병합 후 행 수 | 미매칭 | 판정 |
| --- | --- | ---: | ---: | ---: | --- |
| 주문 ↔ 주문 특징 | one_to_one | 203 | 203 | 0 | PASS |
| 주문 ↔ 고객 | many_to_one | 203 | 203 | 0 | PASS |

### 결과 관찰

불러온 주문 상세 764행에 `line_total = quantity × unit_price`를 대조한 결과 불일치는 없었다. quantity나 unit_price가 0 이하인 행, 필수값 결측, 기본키 중복도 나오지 않았다.

날짜 오류 주문을 제외하면서 상세 데이터는 764행에서 624행으로 줄었고 그중 최종 완료/취소 주문 203건에 딸린 상세는 521행이었다. 주문별로 item_count는 상세 행 수, total_quantity는 수량 합, order_amount는 line_total 합으로 계산했다. 주문별 특징에 order_id 중복은 없었다. 집계를 따로 계산해 공통 함수 결과와 맞춰 보니 세 변수 모두 일치했다.

두 병합 모두 전후 행 수가 203건으로 같았으며 미매칭과 필수 변수 결측도 없었다. 주문 특징 병합에는 `validate='one_to_one'`을 걸었고 주문과 고객을 붙여 날짜를 볼 때는 `validate='many_to_one'`과 `indicator=True`를 썼다. 준비 단계의 외래키 검사(주문→고객, 상세→주문, 상세→상품)도 오류 없이 통과했다.

결과는 `ch10_merge_checks.csv`와 `ch10_data_quality_checks.csv`에 기록했다. 전체 입력의 날짜 오류는 56건(완료/취소 45건)이다. 분석 대상에 날짜 오류가 없는 것은 해당 주문을 제외했기 때문이고 날짜 자체는 고치지 않았다.

### 나의 해석과 판단

병합 전후 행 수가 같고 미매칭도 없었으니 이 과정에서 주문이 중복되거나 필요한 정보가 빠진 흔적은 없었다. 금액과 집계도 따로 계산해 맞춰 봤으므로 기존 결과를 그대로 믿고 넘어갈 때보다 근거가 분명해졌다.

미매칭을 바로 0으로 채우면 실제 금액이나 수량이 0인 경우와 정보를 못 찾은 경우가 뒤섞여 구분할 수 없다. 이번에는 원인을 모르는 값을 0으로 채우지 않았다. 날짜가 맞지 않는 주문은 공통 전처리 데이터에 남겨 두고 `data/processed/ch10/`에 만드는 분석용 입력에서 따로 뺐다. 품질 검사 표의 `QUARANTINED_UNCORRECTED`는 따로 빼 두었을 뿐 날짜는 고치지 않았다는 표시다.

### 업무·분석적 의미

집계나 병합이 틀리면 모델은 잘못된 주문 정보로 학습한다. 모델 성능을 비교하기 전에 계산식과 데이터 연결 방식부터 점검해야 한다는 것을 이번에 알게 됐다. 이번에도 외래키 검사는 통과했지만 날짜 순서 오류는 그 검사로 걸러지지 않았다.

### 한계와 추가 확인 사항

검사를 통과한 범위는 분석에 쓴 203건까지이고 원래 주문 300건 전체의 문제가 해결됐다는 근거는 되지 못한다. 제외한 주문의 날짜를 원천 기록과 대조해야 한다. item_count가 고유 상품 수가 아니라 상세 행 수라는 점도 실제 업무에서 쓸 때는 구분해야 한다.

---

## 4. 클래스 비율

| 클래스 | 의미 | 개수 | 비율 |
| ---: | --- | ---: | ---: |
| 0 | completed | 149 | 73.40% |
| 1 | cancelled | 54 | 26.60% |

![클래스 비율](images/step02_class_ratio.png)

### 결과 관찰

최종 분석 대상 203건의 클래스 분포는 위 표와 같았다.

날짜 오류 주문을 제외하면서 완료 주문은 184건에서 149건으로 35건, 취소 주문은 64건에서 54건으로 10건 줄었다. 그 결과 취소 비율은 25.81%에서 26.60%로 0.79%p 올랐다. 수치는 `ch10_target_distribution.csv`와 `images/step02_class_ratio.png`에서 대조했다.

### 나의 해석과 판단
완료 주문이 취소 주문의 약 2.76배로 클래스가 고르지 않았다. 이런 데이터에서 accuracy는 대부분 완료 주문을 얼마나 맞혔느냐로 정해진다. 전부 완료라고만 예측해도 149/203, 곧 73.40%의 accuracy가 나온다. 70%대 accuracy만 보고는 모델이 무언가를 학습했는지 판단할 수 없다. 반대로 취소 54건을 하나도 못 찾아도 accuracy는 떨어지지 않는다. 클래스 불균형이 accuracy를 실제보다 좋아 보이게 하면서 취소를 얼마나 찾는지는 가려 버리는 셈이다. 그래서 accuracy는 73.40%를 기준선으로 두고 읽고 취소 탐지는 Precision, Recall, F1로 따로 봐야 한다고 판단했다.

### 업무·분석적 의미

취소 위험을 미리 찾는 것이 목적이라면 완료를 많이 맞히는 것 못지않게 실제 취소를 얼마나 잡아내는지도 중요하다. 이에 따라 Dummy 모델을 기준으로 두고 Precision, Recall, F1까지 함께 비교했다.

### 한계와 추가 확인 사항

이 비율은 환불 주문과 날짜 오류로 제외한 주문을 빼고 낸 값이다. 26.60%를 전체 주문이나 앞으로 들어올 주문의 취소율로 볼 수는 없다. 원천 날짜를 대조해 다시 분석하게 되면 그때 클래스 비율과 모델 성능을 함께 비교해야 한다.

---

## 6. train, validation, test 분리

모델과 임계값은 validation에서 선택하고, test는 최종 평가에 한 번만 사용합니다.


In [9]:
from src.classification import (
    split_train_validation_test,
    build_split_summary,
)

(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test:', X_test.shape)
display(split_summary)


X_train: (121, 10)
X_valid: (41, 10)
X_test: (41, 10)


,split,is_cancelled,class_label,count,ratio
0,train,0,completed,89,0.7355
1,train,1,cancelled,32,0.2645
2,validation,0,completed,30,0.7317
3,validation,1,cancelled,11,0.2683
4,test,0,completed,30,0.7317
5,test,1,cancelled,11,0.2683


In [10]:
# 분할 계약 검증 (Test 예측/성능 평가는 수행하지 않음)
train_ix=set(X_train.index);valid_ix=set(X_valid.index);test_ix=set(X_test.index)
split_contract=pd.DataFrame([
 {'check':'train_validation_overlap','count':len(train_ix & valid_ix)},
 {'check':'train_test_overlap','count':len(train_ix & test_ix)},
 {'check':'validation_test_overlap','count':len(valid_ix & test_ix)},
 {'check':'unassigned_rows','count':len(set(model_data.index)-(train_ix|valid_ix|test_ix))},
 {'check':'splits_missing_class','count':sum(set(y.unique())!={0,1} for y in [y_train,y_valid,y_test])}])
assert split_contract['count'].eq(0).all()
split_summary.to_csv(REPORT_DIR/'ch10_split_summary.csv',index=False,encoding='utf-8-sig')
split_contract.to_csv(REPORT_DIR/'ch10_split_contract_checks.csv',index=False,encoding='utf-8-sig')
print('분할 계약');display(split_contract)
print('실제 분할 비중:', {name:round(len(x)/len(model_data),8) for name,x in [('train',X_train),('validation',X_valid),('test',X_test)]})
customer_sets=[set(model_data.loc[x.index,'customer_id']) for x in [X_train,X_valid,X_test]]
print('분할 간 고객 중복(누수 확정이 아닌 비독립성 위험):',{'train_validation':len(customer_sets[0]&customer_sets[1]),'train_test':len(customer_sets[0]&customer_sets[2]),'validation_test':len(customer_sets[1]&customer_sets[2])})


분할 계약


,check,count
0,train_validation_overlap,0
1,train_test_overlap,0
2,validation_test_overlap,0
3,unassigned_rows,0
4,splits_missing_class,0


실제 분할 비중: {'train': 0.59605911, 'validation': 0.20197044, 'test': 0.20197044}
분할 간 고객 중복(누수 확정이 아닌 비독립성 위험): {'train_validation': 23, 'train_test': 21, 'validation_test': 15}


## 7. 기준 모델과 학습 모델 비교

가장 많은 클래스만 예측하는 Dummy 모델을 기준으로 Logistic Regression과 Random Forest의 validation 성능을 비교합니다.


In [11]:
from src.classification import train_and_compare_on_validation

(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

validation_comparison


,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.658537,0.428571,0.818182,0.562500
1,Random Forest,validation,0.707317,0.333333,0.090909,0.142857
2,Dummy Most Frequent,validation,0.731707,0.000000,0.000000,0.000000


In [12]:
# Validation에서 모델만 선택. Threshold/Final Test는 다음 단계에서 평가
from sklearn.metrics import confusion_matrix
candidate_comparison=validation_comparison.loc[validation_comparison['model'].ne('Dummy Most Frequent')].sort_values(['f1','recall','precision'],ascending=False,kind='stable')
selected_model_name=str(candidate_comparison.iloc[0]['model'])
print('Validation 기준 선택 모델:',selected_model_name)
validation_errors=[]
for name in validation_comparison['model']:
    tn,fp,fn,tp=confusion_matrix(y_valid,validation_predictions[name],labels=[0,1]).ravel()
    validation_errors.append({'model':name,'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'threshold':0.5 if name!='Dummy Most Frequent' else None})
validation_errors=pd.DataFrame(validation_errors)
print('Validation 예측 상세 (기본 분류 기준)');display(validation_errors)
fit_checks=[]
for name,pipe in models.items():
    pre=pipe.named_steps['preprocessor'];num=pre.named_transformers_['num'];cat=pre.named_transformers_['cat']
    scale=num.named_steps['scaler'];imp=num.named_steps['imputer']
    train_numeric=X_train[numeric_features].fillna(X_train[numeric_features].median())
    fit_checks.append({'model':name,'train_rows_seen':int(np.asarray(scale.n_samples_seen_).max()),'train_only_scaler':bool(np.allclose(scale.mean_,train_numeric.mean().to_numpy())),'train_only_numeric_imputer':bool(np.allclose(imp.statistics_,X_train[numeric_features].median().to_numpy())),'categorical_strategy':cat.named_steps['imputer'].strategy,'unknown_category_policy':cat.named_steps['onehot'].handle_unknown})
    assert fit_checks[-1]['train_rows_seen']==len(X_train) and fit_checks[-1]['train_only_scaler'] and fit_checks[-1]['train_only_numeric_imputer']
fit_checks=pd.DataFrame(fit_checks)
print('학습된 전처리와 Train 통계 대조');display(fit_checks)
print('Pipeline 구성:',repr(models[selected_model_name]))
validation_comparison.to_csv(REPORT_DIR/'ch10_validation_model_comparison.csv',index=False,encoding='utf-8-sig')
validation_errors.to_csv(REPORT_DIR/'ch10_validation_prediction_counts.csv',index=False,encoding='utf-8-sig')
fit_checks.to_csv(REPORT_DIR/'ch10_pipeline_fit_checks.csv',index=False,encoding='utf-8-sig')
stage_validation=pd.DataFrame([
 {'check':'binary_target_contract','status':'PASS','detail':'203 rows, completed=149, cancelled=54'},
 {'check':'forbidden_feature_overlap','status':'PASS','detail':'0 forbidden inputs'},
 {'check':'strict_merge_contract','status':'PASS','detail':'analysis: 203 -> 203, unmatched=0 twice'},
 {'check':'temporal_quarantine_contract','status':'PASS' if quarantine_contract_ok else 'FAIL','detail':'56 source orders quarantined (45 binary); prepared input matches Notebook; original dates not repaired'},
 {'check':'analysis_data_quality','status':'PASS','detail':'calculation, keys, features, dates verified on restricted cohort'},
 {'check':'all_splits_have_two_classes','status':'PASS','detail':'train=121, validation=41, test=41; disjoint order rows'},
 {'check':'train_only_preprocessing','status':'PASS','detail':'fit statistics match Train; rows seen=121 for each model'},
 {'check':'selected_model_from_validation','status':'PASS','detail':selected_model_name+' selected by F1, Recall, Precision'},
 {'check':'selected_threshold_from_validation','status':'NOT_RUN','detail':'not run at the model-comparison stage'},
 {'check':'public_prediction_privacy','status':'NOT_RUN','detail':'no public predictions generated'},
 {'check':'test_rows_for_metrics','status':'NOT_RUN','detail':'Test labels used for split summary only; no Test predictions or metrics'}])
stage_validation.to_csv(REPORT_DIR/'ch10_classification_validation.csv',index=False,encoding='utf-8-sig')
print('모델 비교 단계의 검증 (후속 Threshold/Test는 아직 미실행)');display(stage_validation)
for name in ['ch10_target_distribution.csv','ch10_feature_audit.csv','ch10_merge_checks.csv','ch10_data_quality_checks.csv','ch10_split_summary.csv','ch10_validation_model_comparison.csv','ch10_classification_validation.csv','ch10_input_scope_checks.csv','ch10_split_contract_checks.csv','ch10_pipeline_fit_checks.csv','ch10_validation_prediction_counts.csv']:
    print(name,'OK' if (REPORT_DIR/name).is_file() else 'MISSING')


Validation 기준 선택 모델: Logistic Regression
Validation 예측 상세 (기본 분류 기준)


,model,TN,FP,FN,TP,threshold
0,Logistic Regression,18,12,2,9,0.5
1,Random Forest,28,2,10,1,0.5
2,Dummy Most Frequent,30,0,11,0,NaN


학습된 전처리와 Train 통계 대조


,model,train_rows_seen,train_only_scaler,train_only_numeric_imputer,categorical_strategy,unknown_category_policy
0,Dummy Most Frequent,121,True,True,most_frequent,ignore
1,Logistic Regression,121,True,True,most_frequent,ignore
2,Random Forest,121,True,True,most_frequent,ignore


Pipeline 구성: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'item_count',
                                                   'total_quantity',
                                                   'order_amount',
                                                   'order_month',
                                                   'order_dayofweek',
                                                   'days_since_signup']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',

,check,status,detail
0,binary_target_contract,PASS,"203 rows, completed=149, cancelled=54"
1,forbidden_feature_overlap,PASS,0 forbidden inputs
2,strict_merge_contract,PASS,"analysis: 203 -> 203, unmatched=0 twice"
3,temporal_quarantine_contract,PASS,56 source orders quarantined (45 binary); prep...
4,analysis_data_quality,PASS,"calculation, keys, features, dates verified on..."
5,all_splits_have_two_classes,PASS,"train=121, validation=41, test=41; disjoint or..."
6,train_only_preprocessing,PASS,fit statistics match Train; rows seen=121 for ...
7,selected_model_from_validation,PASS,"Logistic Regression selected by F1, Recall, Pr..."
8,selected_threshold_from_validation,NOT_RUN,not run at the model-comparison stage
9,public_prediction_privacy,NOT_RUN,no public predictions generated


ch10_target_distribution.csv OK
ch10_feature_audit.csv OK
ch10_merge_checks.csv OK
ch10_data_quality_checks.csv OK
ch10_split_summary.csv OK
ch10_validation_model_comparison.csv OK
ch10_classification_validation.csv OK
ch10_input_scope_checks.csv OK
ch10_split_contract_checks.csv OK
ch10_pipeline_fit_checks.csv OK
ch10_validation_prediction_counts.csv OK


## 5. Dummy Baseline과 Validation 모델 비교

| 모델 | Accuracy | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Dummy Most Frequent | 0.73 | 0.00 | 0.00 | 0.00 |
| Logistic Regression | 0.66 | 0.43 | 0.82 | 0.56 |
| Random Forest | 0.71 | 0.33 | 0.09 | 0.14 |

![Baseline 비교](images/step03_baseline.png)

- Validation에서 선택한 모델: Logistic Regression

### 결과 관찰

Train 121건으로 학습한 뒤 완료 30건과 취소 11건으로 이루어진 Validation 41건에서 세 모델을 비교했다. Logistic Regression과 Random Forest에는 기본 분류 기준 0.5를 적용했으므로 최종 Threshold를 고르기 전 결과다.

Dummy와 비교하면 지표가 이렇게 달라졌다.

| Dummy 대비 변화 | Accuracy | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Logistic Regression | -7.32%p | +42.86%p | +81.82%p | +0.56 |
| Random Forest | -2.44%p | +33.33%p | +9.09%p | +0.14 |

Dummy는 취소를 0건 예측해 실제 취소 11건을 모두 놓쳤다. 취소 예측이 없으니 Precision을 계산할 분모가 0이 되어 평가 결과에는 0으로 처리됐다. Logistic Regression은 취소 11건 중 9건을 찾고 2건을 놓쳤으며 완료 주문 12건은 취소로 잘못 예측했다. Dummy와 견주면 탐지는 0→9건(+9), 미탐지는 11→2건(-9), 완료 오탐지는 0→12건(+12)으로 바뀌었고 정분류는 30→27건으로 3건 줄었다. Random Forest가 찾은 취소는 1건, 놓친 취소는 10건이었고 완료 오탐지는 2건이었다.

비교 결과는 `ch10_validation_model_comparison.csv`와 `images/step03_baseline.png`로 살펴봤다. 모델 선택에는 Test 성능을 쓰지 않았다.

### 나의 해석과 판단
모델은 Logistic Regression을 선택했다. Dummy를 뺀 두 모델을 F1, Recall, Precision 순서로 따졌다. Logistic Regression의 F1은 0.56으로 Random Forest의 0.14보다 0.42 높았다. Recall은 0.82 대 0.09, Precision은 0.43 대 0.33이었다.

accuracy만 보면 Dummy가 가장 높다. 그런데 취소는 하나도 찾지 못했다. Random Forest는 Logistic Regression보다 accuracy가 4.88%p 높고 오탐지도 적은 대신 취소 탐지가 9건에서 1건으로 8건 줄었다. 이번에는 취소를 찾는 성능과 예측의 정확성을 함께 보려고 F1이 높은 모델을 택했다.

세 모델 모두 Pipeline으로 구성했다. Logistic Regression은 `class_weight='balanced', max_iter=1000`, Random Forest는 `class_weight='balanced', n_estimators=200, random_state=42`였다. 전처리는 Train에서만 학습했다. LLM 검토에서는 Validation에서는 Dummy보다 F1과 Recall이 높았지만, Test에서는 Accuracy가 0.39로 모두 완료로 예측했을 때의 0.73보다 낮다는 지적이 나왔다. Test에서도 취소 일부는 찾았으므로 모든 지표가 Dummy보다 낮은 것은 아니다. 이 섹션은 Validation 기준으로 비교했고 Test에서 드러난 차이는 뒤의 Final Test에 따로 적었다.

### 업무·분석적 의미

Logistic Regression은 완료만 예측하는 모델보다 취소를 많이 찾은 대신 실제 완료 주문을 위험하다고 표시하는 경우도 늘었다. 취소 탐지의 이점을 얻는 만큼 추가 확인 업무도 생기므로 두 가지를 함께 고려해야 한다.

### 한계와 추가 확인 사항

Validation의 취소 주문이 11건뿐이라 탐지 한 건마다 Recall이 약 9.09%p 달라진다. 분할을 한 번만 했으니 여기서 나온 모델 순위가 계속 유지될지는 모른다. 분석 대상도 날짜가 맞는 203건으로 한정했고 실제 오류 비용은 알지 못하므로, 이 모델이 업무에서도 가장 이득이라고는 단정하지 않았다.

---

## 6. Train / Validation / Test 역할

| Split | 역할 | 클래스 0 건수 | 클래스 1 건수 |
| --- | --- | ---: | ---: |
| Train | 모델 학습 | 89 | 32 |
| Validation | 모델/Threshold 선택 | 30 | 11 |
| Test | 선택 이후 최종 평가 | 30 | 11 |

### 결과 관찰

stratified random split으로 클래스 비율을 유지했고 `random_state=42`로 고정했다.

| Split | 역할 | 전체 건수 | 클래스 0 건수 | 클래스 1 건수 | 클래스 1 비율 | 전체 대비 비중 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Train | 파라미터/전처리 학습 | 121 | 89 | 32 | 26.45% | 59.61% |
| Validation | 모델 선택 및 이후 Threshold 선택 | 41 | 30 | 11 | 26.83% | 20.20% |
| Test | 모든 선택 고정 이후 최종 평가 | 41 | 30 | 11 | 26.83% | 20.20% |
| 합계 |  | 203 | 149 | 54 | 26.60% | 100% |

전체 취소 비율 26.60%보다 Train은 0.15%p 낮았고 Validation과 Test는 각각 0.23%p 높았다. 주문 수를 정수로 나누다 보니 비율이 완전히 같지는 않았다. 그래도 세 분할 모두에 두 클래스가 들어 있었다.

Train과 Validation, Train과 Test, Validation과 Test 사이의 주문 행 중복은 각각 0건이었다. 미배정 주문은 0건이었고 한 클래스가 없는 분할도 0개였다. 클래스별 건수를 합치면 완료 149건, 취소 54건으로 원래 분석 대상과 같았다. 세 모델의 scaler가 학습한 행 수는 모두 121건이었고 Train 평균 및 중앙값과도 맞아떨어졌다. 결과는 `ch10_split_summary.csv`에 저장했다.

### 나의 해석과 판단
Train은 모델과 전처리 학습에, Validation은 모델과 Threshold 선택에, Test는 선택이 끝난 뒤 최종 평가에 썼다. Test를 보면서 선택을 바꾸면 그 데이터에 맞춰 모델을 고르는 셈이 된다. 그러면 새로운 주문에서도 잘 맞힐지 공정하게 평가하기 어렵다.

취소 주문이 적어서 어느 분할에서 취소가 빠지거나 비율이 크게 틀어질 수 있었고 이를 줄이려고 층화 분할을 사용했다. 이번 비교에는 일반 무작위 분할보다 적절하다고 판단했다. 다만 실제 시간 순서를 반영하지 못한 평가라는 한계는 남는다.

### 교육용 random split의 한계
실제 미래 주문 예측 운영 전에 추가해야 할 평가를 작성하세요.

미래 주문을 예측하려면 시간순 검증과 별도의 out-of-time 평가가 필요하다. 시간순 검증은 과거 자료로 학습하고 이후 시기의 자료로 평가하는 방식인데, 이번에는 이 평가들을 수행하지 않았다. 날짜 오류 주문을 제외한 처리와 주문 당시 정보인지에 관한 가정도 그대로 남아 있다. 분석용 입력과 Notebook, 스크립트의 일치 검사는 통과했으나(뒤의 `전체 파이프라인 실행과 산출물 저장` 단계에서 확인) 미래 성능은 이 검사가 보장하는 범위 밖이다.

### 업무·분석적 의미

같은 Validation으로 후보 모델을 비교해야 평가 대상이 달라서 생기는 차이를 줄일 수 있다. 이번에는 Test 결과를 모델과 Threshold 선택에 쓰지 않았고 전체 재실행에서도 같은 데이터 처리와 분할 기준을 유지했다. (전체 재실행은 뒤의 `전체 파이프라인 실행과 산출물 저장` 단계에서 확인)

### 한계와 추가 확인 사항

주문 행은 겹치지 않았지만 같은 고객은 Train/Validation 사이 23명, Train/Test 사이 21명, Validation/Test 사이 15명이 겹쳤다. 이 겹침만으로 누수라고 단정하지는 않았다. 다만 고객별 패턴 때문에 결과가 낙관적으로 나왔을 가능성은 열어 두었다. 신규 고객의 주문 예측이 목적이라면 고객 단위로 나누는 방법도 필요하다.

---

## 8. validation에서 모델과 임계값 선택

Dummy 모델은 기준선이므로 실제 선택 후보에서는 제외합니다. validation의 f1-score를 우선하고 recall과 precision을 함께 확인합니다.


In [13]:
from src.classification import threshold_metrics, choose_threshold

non_dummy_comparison = validation_comparison[
    validation_comparison['model'] != 'Dummy Most Frequent'
]

selected_model_name = str(non_dummy_comparison.iloc[0]['model'])
selected_model = models[selected_model_name]
validation_proba = validation_probabilities[selected_model_name]

threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print('선택 모델:', selected_model_name)
print('선택 임계값:', selected_threshold)
display(threshold_df.sort_values(['f1', 'recall'], ascending=False).head(10))


선택 모델: Logistic Regression
선택 임계값: 0.45


,threshold,accuracy,precision,recall,f1
5,0.45,0.658537,0.428571,0.818182,0.562500
6,0.50,0.658537,0.428571,0.818182,0.562500
4,0.40,0.634146,0.409091,0.818182,0.545455
8,0.60,0.731707,0.500000,0.545455,0.521739
7,0.55,0.682927,0.437500,0.636364,0.518519
2,0.30,0.463415,0.322581,0.909091,0.476190
3,0.35,0.512195,0.333333,0.818182,0.473684
9,0.65,0.780488,0.666667,0.363636,0.470588
0,0.20,0.341463,0.289474,1.000000,0.448980
1,0.25,0.365854,0.285714,0.909091,0.434783


In [14]:
# Validation 결과만으로 선택 고정 후 Evidence 저장
from datetime import datetime, timezone, timedelta
import hashlib, json
from src.classification import evaluate_classification_predictions
assert selected_model_name == 'Logistic Regression'
validation_selected_metrics=evaluate_classification_predictions(y_valid,(validation_proba>=selected_threshold).astype(int))
print('전체 Threshold 비교');display(threshold_df.sort_values('threshold'))
print('선택 Threshold에서 Validation 성능');display(pd.DataFrame([validation_selected_metrics]))
threshold_df.to_csv(REPORT_DIR/'ch10_validation_threshold_metrics.csv',index=False,encoding='utf-8-sig')
freeze_record={'selected_model_name':selected_model_name,'selected_threshold':float(selected_threshold),'selection_data':'validation only','n_train':len(X_train),'n_validation':len(X_valid),'n_test':len(X_test),'model_rule':'F1, Recall, Precision; non-Dummy','threshold_rule':'choose_threshold(validation)','frozen_at':datetime.now(timezone(timedelta(hours=9))).isoformat(),'test_evaluated_before_freeze':False,'analysis_rows':len(model_data)}
(REPORT_DIR/'ch10_selection_freeze.json').write_text(json.dumps(freeze_record,ensure_ascii=False,indent=2),encoding='utf-8')
frozen_choice=(selected_model_name,float(selected_threshold))
print('Test 확인 전 선택 고정:',json.dumps(freeze_record,ensure_ascii=False))


전체 Threshold 비교


,threshold,accuracy,precision,recall,f1
0,0.20,0.341463,0.289474,1.000000,0.448980
1,0.25,0.365854,0.285714,0.909091,0.434783
2,0.30,0.463415,0.322581,0.909091,0.476190
3,0.35,0.512195,0.333333,0.818182,0.473684
4,0.40,0.634146,0.409091,0.818182,0.545455
5,0.45,0.658537,0.428571,0.818182,0.562500
6,0.50,0.658537,0.428571,0.818182,0.562500
7,0.55,0.682927,0.437500,0.636364,0.518519
8,0.60,0.731707,0.500000,0.545455,0.521739
9,0.65,0.780488,0.666667,0.363636,0.470588


선택 Threshold에서 Validation 성능


,accuracy,precision,recall,f1
0,0.658537,0.428571,0.818182,0.5625


Test 확인 전 선택 고정: {"selected_model_name": "Logistic Regression", "selected_threshold": 0.45, "selection_data": "validation only", "n_train": 121, "n_validation": 41, "n_test": 41, "model_rule": "F1, Recall, Precision; non-Dummy", "threshold_rule": "choose_threshold(validation)", "frozen_at": "2026-11-15T13:33:03.883088+09:00", "test_evaluated_before_freeze": false, "analysis_rows": 203}


In [15]:
# Threshold별 실제 Validation 오류 수 (Test와 독립)
threshold_counts=[]
for t in threshold_df['threshold']:
    tn,fp,fn,tp=confusion_matrix(y_valid,(validation_proba>=t).astype(int),labels=[0,1]).ravel()
    threshold_counts.append({'threshold':float(t),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)})
threshold_counts=pd.DataFrame(threshold_counts)
threshold_comparison=threshold_df.merge(threshold_counts,on='threshold',validate='one_to_one')
threshold_comparison.to_csv(REPORT_DIR/'ch10_validation_threshold_counts.csv',index=False,encoding='utf-8-sig')
print('Validation Threshold별 실제 결과');display(threshold_comparison)
print('0.45 vs 0.50 Validation 예측 차이:',int(np.count_nonzero((validation_proba>=0.45)!=(validation_proba>=0.50))))


Validation Threshold별 실제 결과


,threshold,accuracy,precision,recall,f1,TN,FP,FN,TP
0,0.20,0.341463,0.289474,1.000000,0.448980,3,27,0,11
1,0.25,0.365854,0.285714,0.909091,0.434783,5,25,1,10
2,0.30,0.463415,0.322581,0.909091,0.476190,9,21,1,10
3,0.35,0.512195,0.333333,0.818182,0.473684,12,18,2,9
4,0.40,0.634146,0.409091,0.818182,0.545455,17,13,2,9
5,0.45,0.658537,0.428571,0.818182,0.562500,18,12,2,9
6,0.50,0.658537,0.428571,0.818182,0.562500,18,12,2,9
7,0.55,0.682927,0.437500,0.636364,0.518519,21,9,4,7
8,0.60,0.731707,0.500000,0.545455,0.521739,24,6,5,6
9,0.65,0.780488,0.666667,0.363636,0.470588,28,2,7,4


0.45 vs 0.50 Validation 예측 차이: 0


## 7. Validation 성능과 우선 지표

- Accuracy: 0.66
- Precision: 0.43
- Recall: 0.82
- F1: 0.56

### 결과 관찰

선택한 Logistic Regression에 Threshold 0.45를 적용해 Validation 결과를 살펴봤다. Validation은 완료 30건과 취소 11건을 합쳐 총 41건이다.

기본값 0.50으로 계산해도 예측이 똑같아 네 지표 역시 같았다. 실제 취소 11건 가운데 9건을 찾았고 2건을 놓쳤다. 취소로 예측한 주문 21건 중 실제 취소는 9건이었다. 수치는 `ch10_validation_model_comparison.csv`와 `ch10_validation_threshold_metrics.csv`에서 대조했다.

### 내가 가장 중요하게 본 지표

나는 F1을 우선으로 두고 Recall도 함께 중요하게 봤다.

### 그 이유

F1은 Precision과 Recall의 조화평균이라 취소를 얼마나 찾는지와 취소 예측을 얼마나 믿을 만한지를 같이 살펴볼 수 있다. 완료 비율이 높은 데이터에서는 accuracy만 보는 것보다 이쪽이 목적에 맞는다고 봤다.

> 실제 FP/FN 비용 정보가 없다면 가정임을 명시하세요.

### 나의 해석과 판단

취소 위험을 놓쳐서 생기는 손실이 추가 확인 비용보다 클 수 있다고 가정하고 Recall도 중시했다. 다만 실제 비용은 알지 못하므로 FN이 항상 더 비싸다고 못 박지는 않았다. 확인 인력이 부족하거나 오탐지 비용이 큰 상황이라면 Precision을 더 중시하는 쪽을 택할 수도 있다.

### 업무·분석적 의미

Precision이 0.43이므로 위험하다고 표시한 주문 가운데 절반 이상은 실제 취소가 아니었다. 도입을 정할 때는 취소 탐지가 나아졌다는 점과 함께 추가 확인 업무를 감당할 수 있는지도 살펴봐야 한다.

### 한계와 추가 확인 사항

취소 표본이 11건으로 작다. 실제 비용, 개입 효과, 확률 보정도 확인하지 못했다. 이 수치는 Validation 결과라서 최종 Test 결과와 구분해 해석했다. 취소가 11건뿐이니 한 건만 더 찾거나 놓쳐도 Recall이 약 9.09%p 달라지고 0.82라는 값도 그만큼 흔들릴 수 있다.

---

## 8. Threshold 비교

| Threshold | Precision | Recall | F1 | 예상 특징 |
| ---: | ---: | ---: | ---: | --- |
| 0.30 | 0.32 | 0.91 | 0.48 | 미탐지 1건, 오탐지 다수 |
| 0.45 | 0.43 | 0.82 | 0.56 | 선택값, 최대 F1 동률 |
| 0.60 | 0.50 | 0.55 | 0.52 | 오탐지 감소와 취소 미탐지 증가 |

![Threshold 비교](images/step05_threshold.png)

### 결과 관찰

Logistic Regression의 Validation 확률로 Threshold 0.20~0.80을 0.05 간격으로 비교했다. 총 13개 후보 중 대표 결과를 아래 표에 추렸다.

| Threshold | Precision | Recall | F1 | FP | FN | 관찰한 특징 |
| ---: | ---: | ---: | ---: | ---: | ---: | --- |
| 0.20 | 0.29 | 1.00 | 0.45 | 27 | 0 | 취소 11건 전부 탐지, 오탐지 27건 |
| 0.30 | 0.32 | 0.91 | 0.48 | 21 | 1 | 미탐지 1건, 오탐지 다수 |
| 0.40 | 0.41 | 0.82 | 0.55 | 13 | 2 | 0.45와 탐지 수는 같지만 오탐지 1건 많음 |
| **0.45** | **0.43** | **0.82** | **0.56** | **12** | **2** | 선택값, 최대 F1 동률 |
| 0.50 | 0.43 | 0.82 | 0.56 | 12 | 2 | 0.45와 예측 0건 차이 |
| 0.60 | 0.50 | 0.55 | 0.52 | 6 | 5 | 오탐지 감소와 취소 미탐지 증가 |
| 0.65 | 0.67 | 0.36 | 0.47 | 2 | 7 | 높은 정밀도, 낮은 탐지율 |
| 0.80 | 1.00 | 0.18 | 0.31 | 0 | 9 | 취소 예측 2건 모두 맞지만 9건 놓침 |

0.60에서 0.45로 낮추면 Precision은 0.50→0.43(-0.07), Recall은 0.55→0.82(+0.27), F1은 0.52→0.56(+0.04)으로 변했다. FP는 6→12건(+6), FN은 5→2건(-3)이었다. 반면 0.50에서 0.45로 바꿨을 때는 네 지표와 FP, FN의 변화가 모두 0이었다.

전체 비교는 `ch10_validation_threshold_metrics.csv`에 저장했다. 그래프는 화면을 캡처해 `images/step05_threshold.png`로 첨부했다.

### 내가 선택한 Threshold

최종 Threshold는 **0.45**로 정했다.

### 선택 이유

Validation F1이 0.56으로 가장 높았고 F1이 같은 후보끼리는 Recall과 Precision도 함께 따져 봤다. `choose_threshold`가 F1, Recall, Precision을 기준으로 비교해 낸 결과다. 0.45와 0.50은 세 지표도 예측도 같으므로 0.45 쪽이 더 좋은 결과를 냈다고 하기는 어렵다. 0.50을 유지하는 방법도 있었지만 이번에는 함수가 선택한 값을 기록해 그대로 사용했다.

0.45는 0.60보다 취소를 더 많이 찾으므로, 확인 업무가 늘더라도 취소 누락을 줄이려는 목적에는 맞는다고 봤다. 0.20은 누락이 0건인 대신 오탐지가 27건이라 확인 부담이 크다. 실제 비용 자료가 없어서 업무 비용까지 가장 적게 만드는 값이 어느 것인지는 판단하지 않았다.

### 나의 해석과 판단
선택은 Validation 단계에서 마쳤다. 모델과 Threshold를 먼저 기록해 둔 뒤에야 Test를 열어 봤다. Test 결과를 보고 다시 고르면 최종 평가에 쓸 자료까지 선택에 끌어다 쓰는 셈이 된다.

### 업무·분석적 의미

Threshold를 낮추거나 높이는 일은 위험 주문을 어디까지 표시할지 정하는 과정이었다. Validation에서 0.60을 쓰면 위험으로 표시되는 주문이 12건이고 0.45에서는 21건으로 늘어난다. 그 대신 놓치는 취소는 5건에서 2건으로 준다. Threshold는 지표와 더불어 확인 업무를 몇 건까지 감당할 수 있는지까지 놓고 정해야 한다고 봤다.

### 한계와 추가 확인 사항

다른 기간에서도 0.45가 적절한지, 예측 확률이 실제 취소 가능성과 잘 맞는지는 추가 확인이 필요하다. Validation의 취소가 11건뿐이어서 0.45와 0.50처럼 예측이 똑같이 나오는 구간도 생겼다. 표본이 달라지면 가장 좋은 Threshold도 바뀔 수 있다.

---

## 9. test 데이터 최종 평가

이제 선택이 끝났으므로 test 데이터에서 최종 성능을 한 번 확인합니다.


In [16]:
from src.classification import (
    final_test_evaluation,
    confusion_matrix_dataframe,
    classification_report_dataframe,
)

y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)
classification_report_df = classification_report_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)
display(classification_report_df)


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.45,frozen_before_test,0.390244,0.208333,0.454545,0.285714


,pred_completed,pred_cancelled
actual_completed,11,19
actual_cancelled,6,5


,label,precision,recall,f1-score,support
0,completed,0.647059,0.366667,0.468085,30.000000
1,cancelled,0.208333,0.454545,0.285714,11.000000
2,accuracy,0.390244,0.390244,0.390244,0.390244
3,macro avg,0.427696,0.410606,0.376900,41.000000
4,weighted avg,0.529352,0.390244,0.419156,41.000000


In [17]:
# 고정된 모델·Threshold의 1차 Final Test 및 비교
assert frozen_choice==(selected_model_name,float(selected_threshold))
assert freeze_record['test_evaluated_before_freeze'] is False
primary_test_predictions=y_pred_test.copy();primary_test_probabilities=y_proba_test.copy()
print('Test metrics schema:',test_metrics.columns.tolist());print(test_metrics.to_string(index=False))
print('Confusion Matrix');print(confusion_df.to_string())
print('Classification Report');print(classification_report_df.to_string())
print('Validation selected:',validation_selected_metrics)
print('Final Test TN FP FN TP:',confusion_matrix(y_test,y_pred_test,labels=[0,1]).ravel().tolist())


Test metrics schema: ['evaluation_split', 'threshold', 'selection_status', 'accuracy', 'precision', 'recall', 'f1']
evaluation_split  threshold   selection_status  accuracy  precision   recall       f1
            test       0.45 frozen_before_test  0.390244   0.208333 0.454545 0.285714
Confusion Matrix
                  pred_completed  pred_cancelled
actual_completed              11              19
actual_cancelled               6               5
Classification Report
          label  precision    recall  f1-score    support
0     completed   0.647059  0.366667  0.468085  30.000000
1     cancelled   0.208333  0.454545  0.285714  11.000000
2      accuracy   0.390244  0.390244  0.390244   0.390244
3     macro avg   0.427696  0.410606  0.376900  41.000000
4  weighted avg   0.529352  0.390244  0.419156  41.000000
Validation selected: {'accuracy': 0.6585365853658537, 'precision': 0.42857142857142855, 'recall': 0.8181818181818182, 'f1': 0.5625}
Final Test TN FP FN TP: [11, 19, 6, 5]


In [18]:
# 같은 모델·Threshold에서 Validation 대비 Test 변화
from src.classification import evaluate_classification_predictions
validation_selected_metrics=evaluate_classification_predictions(y_valid,(validation_proba>=selected_threshold).astype(int))
metric_delta=pd.DataFrame([{'metric':m,'validation':validation_selected_metrics[m],'test':float(test_metrics[m].iloc[0]),'test_minus_validation':float(test_metrics[m].iloc[0])-validation_selected_metrics[m]} for m in ['accuracy','precision','recall','f1']])
metric_delta.to_csv(REPORT_DIR/'ch10_validation_test_changes.csv',index=False,encoding='utf-8-sig')
print('동일 선택의 Validation -> Test');display(metric_delta)


동일 선택의 Validation -> Test


,metric,validation,test,test_minus_validation
0,accuracy,0.658537,0.390244,-0.268293
1,precision,0.428571,0.208333,-0.220238
2,recall,0.818182,0.454545,-0.363636
3,f1,0.562500,0.285714,-0.276786


## 9. FP/FN 해석

- FP 의미: 완료 주문을 취소 위험으로 잘못 예측한 경우 (Test 19건). 취소하지 않을 주문에도 확인 연락이나 점검을 하게 돼 불필요한 일이 늘 수 있다.
- FN 의미: 취소 주문을 완료로 놓친 경우 (Test 6건). 취소될 주문을 미리 살펴볼 기회를 놓친다.
- 현재 목적에서 더 부담스러울 수 있는 오류: FN (취소 위험 누락). 실제 비용을 모른 채 세운 가정이다. FP 19건도 무시하기 어렵다.
- 그 판단에 추가로 필요한 업무 정보: FP/FN 건당 비용, 확인에 걸리는 시간과 인력, 취소 손실, 고객 불편, 개입 성공률

![Confusion Matrix](images/step06_confusion_matrix.png)

### 결과 관찰

Logistic Regression과 Threshold 0.45를 고정한 뒤 Test 41건의 혼동행렬을 살펴봤다. 양성은 취소 주문이다.

| 실제 / 예측 | 완료 예측 | 취소 위험 예측 | 합계 |
| --- | ---: | ---: | ---: |
| 완료 | TN=11 | FP=19 | 30 |
| 취소 | FN=6 | TP=5 | 11 |
| 합계 | 17 | 24 | 41 |

완료 주문을 취소 위험으로 잘못 예측한 FP는 19건, 취소 주문을 완료로 놓친 FN은 6건이었다. 완료를 맞힌 TN은 11건, 취소를 찾은 TP는 5건이다. 정분류는 16건, 오류는 25건이었다. 위험으로 표시한 24건 중 실제 취소는 5건이었고 실제 취소 11건 중 6건은 놓쳤다.

같은 조건의 Validation과 비교하면 TN은 18→11건(-7), FP는 12→19건(+7), FN은 2→6건(+4), TP는 9→5건(-4)이었다. 두 데이터의 실제 클래스 수는 각각 완료 30건, 취소 11건으로 같았다. 수치는 `ch10_confusion_matrix.csv`, `ch10_test_metrics.csv`, `images/step06_confusion_matrix.png`에서 대조했다.

### 나의 해석과 판단

FP가 많으면 실제로는 취소하지 않을 주문에까지 확인 연락이나 점검이 들어가 불필요한 일이 늘 수 있다. 반대로 FN은 취소될 주문을 미리 살펴볼 기회를 놓치는 경우다. 이번에는 취소 위험 누락을 줄이는 쪽에 더 무게를 뒀다. 다만 FP 19건도 무시하기 어려웠다.

어느 오류가 더 큰 손해인지는 건당 비용과 대응 효과를 알아야 정할 수 있다. 식으로만 쓰면 현재 오류 비용은 `19×FP 건당 비용 + 6×FN 건당 비용`이다. 비용 자료가 없어서 금액은 계산하지 않았다. 취소 위험을 발견했을 때 실제로 취소를 막을 수 있는지도 아직 모른다.

### 업무·분석적 의미

자동 차단이나 자동 취소 같은 조치를 하기에는 이 결과의 근거가 부족하다. 확인에 걸리는 시간과 인력, 취소 손실, 고객 불편, 개입 성공률을 알아야 오류를 줄이는 것이 실제 업무에 도움이 되는지 판단할 수 있다.

### 한계와 추가 확인 사항

Test에서 FP와 FN이 모두 늘었는데 그 원인을 특정 고객 특성이나 변수로 단정할 수는 없다. 취소 사유나 개입 실험 자료도 없다. 실제 비용도 가정으로만 남겼다.

---

## 10. Final Test

- 고정한 모델: Logistic Regression
- 고정한 Threshold: 0.45
- Accuracy: 0.39
- Precision: 0.21
- Recall: 0.45
- F1: 0.29
- Validation 대비 변화: Accuracy -0.27, Precision -0.22, Recall -0.36, F1 -0.28 (네 지표 모두 하락)

### 결과 관찰

Validation에서 Logistic Regression과 Threshold 0.45를 정한 뒤 Test를 평가했다. Test는 완료 30건과 취소 11건, 총 41건이었다.

| 지표 | Validation | Final Test | Test−Validation |
| --- | ---: | ---: | ---: |
| Accuracy | 0.66 | 0.39 | -0.27 (-26.83%p) |
| Precision | 0.43 | 0.21 | -0.22 (-22.02%p) |
| Recall | 0.82 | 0.45 | -0.36 (-36.36%p) |
| F1 | 0.56 | 0.29 | -0.28 |

혼동행렬은 TN=11, FP=19, FN=6, TP=5였다. Accuracy는 16/41, Precision은 5/24, Recall은 5/11로 계산되고 `classification_report` 결과와도 일치했다. 근거 파일은 `ch10_test_metrics.csv`와 `ch10_confusion_matrix.csv`다.

### 나의 해석과 판단

네 지표 모두 Validation보다 낮았다. 앞에서 본 성능이 Test에서는 유지되지 않은 셈이다. Test Accuracy 0.39는 전부 완료로 예측할 때 나오는 0.73(30/41)보다도 낮다. 현재 결과로는 새로운 주문에도 잘 맞으리라 보기 어렵다. 두 데이터의 클래스 비율이 같으니 단순히 취소 비율 차이로 성능 하락을 설명할 수도 없다.

가능한 이유로는 작은 표본의 구성 차이, Validation 한 번으로 모델을 고른 영향, 고객이나 입력값 분포의 차이 등을 꼽을 수 있다. 어느 설명이 맞는지는 추가 분석을 하지 않아 원인으로 단정하지 않았다.

### 선택 보호 확인
- [x] Test 결과를 보기 전에 모델을 고정했습니다.
- [x] Test 결과를 보기 전에 Threshold를 고정했습니다.
- [x] Test 결과를 본 뒤 같은 Test를 이용해 선택을 바꾸지 않았습니다.

전체 재실행 때도 같은 203건과 분할 기준을 써서 모델, Threshold, 예측, 지표가 그대로 나왔다. (전체 재실행은 뒤의 `전체 파이프라인 실행과 산출물 저장` 단계에서 확인) 재실행은 같은 Test를 다시 계산한 것이므로 결과가 재현되는지 점검하는 용도로만 봤다.

### 업무·분석적 의미

코드를 다시 실행해 같은 값이 나온 것과 모델 성능이 충분한지는 별개다. 실제 사용 여부는 Validation의 높은 수치만이 아니라 Test에서 드러난 오류와 대응 비용까지 보고 판단해야 한다.

### 한계와 추가 확인 사항

Test는 41건뿐인 작은 무작위 분할이고 같은 고객이 여러 분할에 걸쳐 있다. 날짜가 맞는 주문만 썼다는 제한도 있다. 앞으로 개선 방법을 고르려면 새로운 Validation 구성과 별도 holdout을 마련해야 한다. 시간에 따른 성능 변화와 확률 보정도 아직 살펴보지 못한 과제로 남았다.

---

혼동행렬은 다음처럼 읽습니다.

| 구분 | 의미 |
|---|---|
| True Negative | 실제 완료 주문을 완료로 예측 |
| False Positive | 실제 완료 주문을 취소로 잘못 예측 |
| False Negative | 실제 취소 주문을 완료로 잘못 예측 |
| True Positive | 실제 취소 주문을 취소로 예측 |


## 10. 개인정보를 제외한 예측 결과 생성

고객명, 이메일, 주소 같은 불필요한 식별정보는 결과 파일에 저장하지 않습니다.


In [19]:
from src.classification import create_prediction_result, public_prediction_result
# source_index는 내부 추적용으로만 유지하고 공개 셀에는 공개 결과만 표시
prediction_result = create_prediction_result(source_index=X_test.index,y_test=y_test,y_pred=y_pred_test,y_proba=y_proba_test,model_name=selected_model_name,threshold=selected_threshold)
prediction_result_internal=prediction_result.copy()
prediction_result_public=public_prediction_result(prediction_result_internal)
public_allowed=['record_id','actual_is_cancelled','predicted_is_cancelled','cancel_probability','model','threshold']
privacy_forbidden={'source_index','order_id','customer_id','product_id','name','email','address'}
assert prediction_result_public.columns.tolist()==public_allowed
assert not privacy_forbidden.intersection(prediction_result_public.columns)
assert len(prediction_result_public)==len(X_test)==41
assert prediction_result_public['record_id'].is_unique
print('내부 컬럼:',prediction_result_internal.columns.tolist())
print('공개 컬럼:',prediction_result_public.columns.tolist())
print('공개 금지 컬럼 수:',len(privacy_forbidden.intersection(prediction_result_public.columns)))
print('공개 행 수:',len(prediction_result_public),'record_id 중복:',int(prediction_result_public['record_id'].duplicated().sum()))
display(prediction_result_public.head())


내부 컬럼: ['record_id', 'source_index', 'actual_is_cancelled', 'predicted_is_cancelled', 'cancel_probability', 'model', 'threshold']
공개 컬럼: ['record_id', 'actual_is_cancelled', 'predicted_is_cancelled', 'cancel_probability', 'model', 'threshold']
공개 금지 컬럼 수: 0
공개 행 수: 41 record_id 중복: 0


,record_id,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,0,1,0.505311,Logistic Regression,0.45
1,test_0002,0,1,0.532967,Logistic Regression,0.45
2,test_0003,0,0,0.412199,Logistic Regression,0.45
3,test_0004,1,1,0.904746,Logistic Regression,0.45
4,test_0005,0,1,0.488737,Logistic Regression,0.45


## 11. Internal / Public 결과 구분

### 공개 Prediction 컬럼

```text
record_id
actual_is_cancelled
predicted_is_cancelled
cancel_probability
model
threshold
```

### 공개 결과에 없어야 할 컬럼

```text
source_index
order_id
customer_id
product_id
```

### 결과 관찰

내부 예측 결과에는 `record_id`, `source_index`, `actual_is_cancelled`, `predicted_is_cancelled`, `cancel_probability`, `model`, `threshold`의 7개 컬럼이 있었다. 이 중 source_index는 원래 행을 다시 찾기 위한 정보다.

공개 결과에서는 source_index를 빼고 위의 6개 컬럼만 사용했다.

공개 결과는 41행이고 record_id 중복은 0건이었다. `source_index`, `order_id`, `customer_id`, `product_id`, `name`, `email`, `address` 중 공개 컬럼에 포함된 것은 0개였다. `ch10_classification_predictions.csv`를 다시 읽어 행 수와 컬럼을 점검했고 저장된 예측과 확률이 평가 결과와 같은지도 대조했다. 개인정보 관련 자동 검사도 PASS였다. (CSV 재확인과 자동 검사는 뒤의 `전체 파이프라인 실행과 산출물 저장` 단계에서 확인)

### 나의 해석과 판단
모델 오류 분석용 내부 자료와 공개 결과를 왜 분리해야 하는지 작성하세요.

틀린 예측을 살펴볼 때는 원래 행을 찾을 수 있는 내부 정보가 필요하다. 하지만 결과를 공개할 때 그 정보까지 보여 줄 필요는 없다고 봤다. 화면에도 내부 결과 대신 공개용 결과를 표시했다. LLM 검토는 record_id가 order_id와 바로 이어지면 고객 정보와 연결될 수 있다고 지적했다. 그런데 record_id는 Test 결과에 붙인 순번이고 원래 행과의 대응은 내부 파일에만 있다. 그래서 공개 결과만으로는 주문을 찾을 수 없다고 판단했다.

### 업무·분석적 의미

내부 오류 분석에 필요한 자료와 외부에 설명할 결과를 나누면 불필요한 식별자 노출을 줄일 수 있다. 모델 성능이 좋더라도 공개 파일에 개인정보가 섞였는지는 따로 점검해야 한다. 이번에는 내부 7개 컬럼에서 `source_index`를 뺀 6개 컬럼만 공개용으로 남겼다. `source_index`가 남아 있으면 내부 자료와 맞춰 원래 주문 행을 다시 찾을 수 있기 때문이다.

### 한계와 추가 확인 사항

record_id는 이 결과 안에서 붙인 순번이라 완전한 익명화를 보장하기에는 부족하다. 다른 자료와 연결했을 때의 위험이나 내부 자료 접근 권한은 추가 검토가 필요하다. Notebook에는 내부 식별자의 실제 행 값을 표시하지 않았다.

---

## 11. LLM 코드 검토 체크리스트


In [20]:
from src.classification import build_classification_checklist
classification_checklist = build_classification_checklist()
checked_status = [
    '확인' if set(model_data['is_cancelled'].unique()) == {0, 1} else '미충족',
    '확인' if model_data['order_status'].isin(['completed', 'cancelled']).all() else '미충족',
    '답안에 설명: 주문 생성 직후, 정보 가용성은 교육용 가정',
    '확인' if extra_quality.loc[extra_quality['check_item'].eq('source_line_total_mismatch'), 'count'].eq(0).all() else '미충족',
    '확인' if merge_checks['row_count_preserved'].all() and merge_checks['unmatched_count'].eq(0).all() else '미충족',
    '확인' if not forbidden.intersection(features) else '미충족',
    '확인' if split_contract['count'].eq(0).all() else '미충족',
    '확인' if freeze_record['selection_data'] == 'validation only' else '미충족',
    '최초 평가 후 동일 선택 재현으로 구분; 사후 재선택 없음',
    '확인' if 'Dummy Most Frequent' in validation_comparison['model'].tolist() else '미충족',
    '답안에 네 지표와 FP/FN의 의미·비용 가정을 설명',
    '확인' if not privacy_forbidden.intersection(prediction_result_public.columns) else '미충족',
    '답안에 시간순·out-of-time 평가 필요성과 고객 중복을 설명',
    '예측 패턴만 해석; 취소 원인·개입 효과는 미확인으로 설명',
]
assert len(checked_status) == len(classification_checklist)
classification_checklist['status'] = checked_status
display(classification_checklist)


,check_item,status
0,completed와 cancelled만 사용해 이진 타깃을 만들었는가?,확인
1,refunded 등 다른 상태를 0 클래스에 섞지 않았는가?,확인
2,예측 시점과 feature 가용성 가정을 설명할 수 있는가?,"답안에 설명: 주문 생성 직후, 정보 가용성은 교육용 가정"
3,line_total = quantity × unit_price를 검증했는가?,확인
4,병합에 validate를 사용하고 미매칭 0건을 확인했는가?,확인
5,"order_status, target, ID, 사후 정보를 feature에서 제외했는가?",확인
6,"train, validation, test를 분리했는가?",확인
7,모델과 threshold는 validation에서 선택했는가?,확인
8,모델과 threshold를 고정한 뒤 test를 한 번만 사용했는가?,최초 평가 후 동일 선택 재현으로 구분; 사후 재선택 없음
9,Dummy 기준 모델과 비교했는가?,확인


## 12. 전체 파이프라인 실행과 산출물 저장

위 과정을 한 번에 다시 실행해 CSV와 Markdown 보고서를 생성합니다.


In [21]:
# 수정한 스크립트와 Notebook이 같은 분석용 입력을 쓰는지 재현
from src.classification import run_classification_analysis
import subprocess
execution_checks=[]
for script in ['prepare_ch10_data.py','run_classification_analysis.py']:
    result=subprocess.run([sys.executable,str(PROJECT_ROOT/'scripts'/script)],cwd=PROJECT_ROOT,capture_output=True,text=True,encoding='utf-8',errors='replace')
    print('SCRIPT',script,'EXIT',result.returncode);print(result.stdout);print(result.stderr)
    execution_checks.append({'step':script,'exit_code':result.returncode,'status':'PASS' if result.returncode==0 else 'FAIL','detail':(result.stderr or result.stdout)[-1400:]})
    result.check_returncode()
# 스크립트는 prepare가 만든 같은 경로를 사용하며 임시 표본·PYTHONPATH 우회가 불필요하다.
classification_result=run_classification_analysis(processed_dir=CH10_PROCESSED_DIR,report_dir=REPORT_DIR/'ch10_reproduction',random_state=42)
pd.testing.assert_frame_equal(classification_result['model_data'],model_data)
assert (classification_result['selected_model_name'],float(classification_result['selected_threshold']))==frozen_choice
replay_test=classification_result['test_metrics']
metric_columns=['accuracy','precision','recall','f1']
assert np.allclose(test_metrics[metric_columns].to_numpy(),replay_test[metric_columns].to_numpy())
# 수정한 스크립트가 저장한 결과도 같은 선택·예측·지표인지 대조
script_test=pd.read_csv(REPORT_DIR/'ch10_test_metrics.csv')
script_predictions=pd.read_csv(REPORT_DIR/'ch10_classification_predictions.csv')
assert np.allclose(test_metrics[metric_columns].to_numpy(),script_test[metric_columns].to_numpy())
assert np.array_equal(script_predictions['predicted_is_cancelled'],primary_test_predictions)
assert np.allclose(script_predictions['cancel_probability'],primary_test_probabilities)
assert script_predictions['model'].eq(frozen_choice[0]).all()
assert script_predictions['threshold'].eq(frozen_choice[1]).all()
execution_checks.append({'step':'prepared_cohort_reproduction','exit_code':0,'status':'PASS','detail':'modified script and Notebook share prepared input; model_data, frozen choice, predictions and four metrics identical'})
execution_validation=pd.DataFrame(execution_checks)
assert execution_validation['status'].eq('PASS').all()
execution_validation.to_csv(REPORT_DIR/'ch10_execution_validation.csv',index=False,encoding='utf-8-sig')
print('전체 실행 검증');display(execution_validation[['step','exit_code','status']])
print('수정한 스크립트 및 공통 함수의 모델·Threshold·예측·Test 지표 차이 0');display(replay_test)
print('Test는 최초 최종 평가 후 고정한 선택의 스크립트/공통 함수 재현에서만 다시 계산했으며 선택을 바꾸지 않았다.')


SCRIPT prepare_ch10_data.py EXIT 0
Chapter10 모델링 입력 준비 완료
입력 Raw 경로: C:\dev\llm-data-analysis-course\data\raw

[관계 검증]
                                               check  invalid_count
  orders.customer_id exists in customers.customer_id              0
      order_items.order_id exists in orders.order_id              0
order_items.product_id exists in products.product_id              0

[Chapter10 입력 범위: 날짜를 변경하지 않고 시간 관계 오류 격리]
order_status  original_count  quarantined_count  prepared_count                                    policy
   cancelled              64                 10              54 exclude_signup_after_order_no_date_repair
   completed             184                 35             149 exclude_signup_after_order_no_date_repair
    refunded              52                 11              41 exclude_signup_after_order_no_date_repair

[저장된 processed 파일]
- C:\dev\llm-data-analysis-course\data\processed\customers_clean.csv
- C:\dev\llm-data-analysis-course\data\processed\pro

,step,exit_code,status
0,prepare_ch10_data.py,0,PASS
1,run_classification_analysis.py,0,PASS
2,prepared_cohort_reproduction,0,PASS


수정한 스크립트 및 공통 함수의 모델·Threshold·예측·Test 지표 차이 0


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.45,frozen_before_test,0.390244,0.208333,0.454545,0.285714


Test는 최초 최종 평가 후 고정한 선택의 스크립트/공통 함수 재현에서만 다시 계산했으며 선택을 바꾸지 않았다.


In [22]:
# 실제 공통 자동 검증과 필수 산출물 저장
from src.classification import save_classification_outputs
analysis_validation=classification_result['validation'].copy()
alignment=(len(prediction_result_public)==len(y_test)==41 and np.array_equal(prediction_result_public['actual_is_cancelled'].to_numpy(),y_test.to_numpy()))
final_role=(test_metrics['evaluation_split'].eq('test').all() and test_metrics['selection_status'].eq('frozen_before_test').all())
unchanged=(frozen_choice==(selected_model_name,float(selected_threshold)))
extra_contract=pd.DataFrame([
 {'check':'test_rows_for_metrics','value':len(prediction_result_public),'status':'PASS' if alignment else 'FAIL'},
 {'check':'test_used_as_final_evaluation','value':bool(final_role),'status':'PASS' if final_role else 'FAIL'},
 {'check':'frozen_choice_unchanged','value':bool(unchanged),'status':'PASS' if unchanged else 'FAIL'}])
analysis_validation=pd.concat([analysis_validation,extra_contract],ignore_index=True)
assert analysis_validation['status'].eq('PASS').all()
# 동일 표본·선택·Test 지표 재현 일치를 확인한 공통 파이프라인의 실제 검증표
print('공통 모듈 분석 계약 검증');display(analysis_validation);print('검증 schema:',analysis_validation.columns.tolist())
validation=analysis_validation.copy()
namecol=next((c for c in ['check_item','check','validation_item'] if c in validation.columns),validation.columns[0])
# 원본 날짜가 정상이라고 선언하지 않고, 준비 단계 격리 계약을 검증한다.
assert quarantine_contract_ok
assert set(prepared_quarantine['order_id'])==set(time_audit.loc[time_audit['signup_after_order'],'order_id'])
policy_check={c:'' for c in validation.columns}
policy_check[namecol]='temporal_quarantine_contract';policy_check['status']='PASS';policy_check['value']=len(prepared_quarantine)
validation=pd.concat([validation,pd.DataFrame([policy_check])],ignore_index=True)
assert validation['status'].eq('PASS').all()

output_paths=save_classification_outputs(model_data=model_data,target_dist=target_dist,feature_audit=feature_audit,merge_checks=merge_checks,data_quality_checks=quality_evidence,split_summary=split_summary,validation_comparison=validation_comparison,threshold_df=threshold_df,test_metrics=test_metrics,prediction_result_internal=prediction_result_internal,prediction_result_public=prediction_result_public,confusion_df=confusion_df,report_df=classification_report_df,validation=validation,checklist=classification_checklist,selected_model_name=selected_model_name,selected_threshold=selected_threshold,report_dir=REPORT_DIR)
analysis_validation.to_csv(REPORT_DIR/'ch10_restricted_contract_validation.csv',index=False,encoding='utf-8-sig')
expected=['ch10_target_distribution.csv','ch10_feature_audit.csv','ch10_merge_checks.csv','ch10_data_quality_checks.csv','ch10_split_summary.csv','ch10_validation_model_comparison.csv','ch10_validation_threshold_metrics.csv','ch10_test_metrics.csv','ch10_classification_predictions.csv','ch10_confusion_matrix.csv','ch10_classification_validation.csv','ch10_classification_summary.md']
evidence_checks=[]
for name in expected:
    f=REPORT_DIR/name
    assert f.is_file() and f.stat().st_size>0
    if f.suffix=='.csv':
        loaded=pd.read_csv(f);rows=len(loaded)
        if name=='ch10_classification_predictions.csv':
            assert loaded.columns.tolist()==public_allowed
            assert not privacy_forbidden.intersection(loaded.columns)
            assert len(loaded)==41
            assert np.allclose(loaded['cancel_probability'],primary_test_probabilities)
            assert np.array_equal(loaded['predicted_is_cancelled'],primary_test_predictions)
    else:rows=None
    evidence_checks.append({'file':name,'exists':True,'rows':rows,'bytes':f.stat().st_size})
evidence_checks=pd.DataFrame(evidence_checks);print('파일 내용·존재 검사');display(evidence_checks)
evidence_checks.to_csv(REPORT_DIR/'ch10_evidence_checks.csv',index=False,encoding='utf-8-sig')
print('최종 통합 검증 (검증된 준비 입력 + 원본 격리 계약)');display(validation)
print('모델·Threshold 변경 없음:',frozen_choice==(selected_model_name,float(selected_threshold)))
print('원본 시간 오류 격리 수 (날짜 미정정):',int(time_audit['signup_after_order'].sum()))


공통 모듈 분석 계약 검증


,check,value,status
0,target_exactly_completed_cancelled,"[0, 1]",PASS
1,forbidden_feature_overlap,0,PASS
2,merge_rows_and_matches,True,PASS
3,all_splits_have_both_classes,True,PASS
4,model_selected_on_validation,True,PASS
5,threshold_selected_on_validation,True,PASS
6,public_prediction_identifier_columns,none,PASS
7,test_rows_for_metrics,41,PASS
8,test_used_as_final_evaluation,True,PASS
9,frozen_choice_unchanged,True,PASS


검증 schema: ['check', 'value', 'status']
파일 내용·존재 검사


,file,exists,rows,bytes
0,ch10_target_distribution.csv,True,2.0,86
1,ch10_feature_audit.csv,True,21.0,1953
2,ch10_merge_checks.csv,True,2.0,177
3,ch10_data_quality_checks.csv,True,30.0,1270
4,ch10_split_summary.csv,True,6.0,229
5,ch10_validation_model_comparison.csv,True,3.0,320
6,ch10_validation_threshold_metrics.csv,True,13.0,1003
7,ch10_test_metrics.csv,True,1.0,185
8,ch10_classification_predictions.csv,True,41.0,2517
9,ch10_confusion_matrix.csv,True,2.0,81


최종 통합 검증 (검증된 준비 입력 + 원본 격리 계약)


,check,value,status
0,target_exactly_completed_cancelled,"[0, 1]",PASS
1,forbidden_feature_overlap,0,PASS
2,merge_rows_and_matches,True,PASS
3,all_splits_have_both_classes,True,PASS
4,model_selected_on_validation,True,PASS
5,threshold_selected_on_validation,True,PASS
6,public_prediction_identifier_columns,none,PASS
7,test_rows_for_metrics,41,PASS
8,test_used_as_final_evaluation,True,PASS
9,frozen_choice_unchanged,True,PASS


모델·Threshold 변경 없음: True
원본 시간 오류 격리 수 (날짜 미정정): 56


## 12. Validation Evidence

다음 Evidence 중 확인한 항목에 체크합니다.

- [x] `ch10_target_distribution.csv`
- [x] `ch10_feature_audit.csv`
- [x] `ch10_merge_checks.csv`
- [x] `ch10_data_quality_checks.csv`
- [x] `ch10_split_summary.csv`
- [x] `ch10_validation_model_comparison.csv`
- [x] `ch10_validation_threshold_metrics.csv`
- [x] `ch10_test_metrics.csv`
- [x] `ch10_confusion_matrix.csv`
- [x] `ch10_classification_validation.csv`
- [x] `ch10_classification_predictions.csv` (추가 확인)
- [x] `ch10_classification_summary.md` (추가 확인)

### 결과 관찰

분석 결과가 실제 파일로 남았는지부터 점검했다. `reports/`의 타깃 분포는 2행, Feature Audit는 21행, 병합 검증은 2행, 데이터 품질 검사는 30행, 분할 요약은 6행이었다. 모델 비교는 3행, Threshold 비교는 13행, Test 지표는 1행, 혼동행렬은 2행이었다. 공개 예측은 41행 6컬럼이며 최종 검증표는 11행이었다. 결과 요약 Markdown도 열어 봤다. 필수 파일의 누락과 빈 파일은 각각 0개였다.

공개 CSV의 행 수, 컬럼, 예측, 확률은 평가 결과와 대조했고 계산, 병합, 분할, 선택 과정의 검사 결과도 살펴봤다.

### 자동 Validation 결과

자동 검증 항목은 이진 타깃, 금지 변수 제외, 병합, 두 클래스 분할, Validation 단계의 모델 및 Threshold 선택, 공개 식별자 제거였다. 여기에 Test 41행의 라벨 정렬, 최종 평가 사용 여부, 선택 유지 여부를 더했다. 이 10개에 날짜 오류 주문이 분석용 입력에서 정확히 제외됐는지 보는 1개를 합친 11개가 모두 PASS였고 FAIL은 0개였다.

최종 `ch10_classification_validation.csv`에는 이 11개가 들어 있다. 앞의 모델 비교 단계 출력에 나온 NOT_RUN은 그 시점에 아직 돌리지 않은 검사를 가리킨다. 최종 결과는 마지막 검증표에서 봤다.

### 자동 PASS가 의미하지 않는 것

자동 검사가 모두 통과해도 실제 업무에서 쓸 수 있다는 뜻은 아니다. 원천 날짜 확인, 제외된 주문의 영향, 미래 시점 성능, 고객 중복, 공정성, 오류 비용과 개입 효과는 추가 확인이 필요하다.

### 나의 해석과 판단

원래 날짜 오류는 모두 56건이고 그중 완료/취소 주문은 45건이었다. 이 주문을 따로 보관하면서 주문 수는 300→244건(-56), 상세는 764→624행(-140)이 됐다. 여기서 환불 41건을 추가로 제외해 203건을 분석했다. 원본은 보존하고 `data/processed/ch10/`에 분석용 입력을 저장했다. 제외한 주문의 목록과 사유는 내부 파일로 따로 남겼고 공개 결과에는 넣지 않는다.

여기서 PASS는 원본과 제외 목록을 비교하고 분석용 입력에 해당 주문이 남아 있지 않은지를 확인한 결과일 뿐, 원본 날짜가 맞는지까지 검증하지는 않았다.

### 전체 재실행 확인

| 실행 | 결과 |
| --- | --- |
| `prepare_ch10_data.py` (수정본) | 종료 코드 0. 날짜 오류 주문을 따로 빼고 분석용 입력을 저장 |
| `run_classification_analysis.py` (수정본) | 종료 코드 0. 위 분석용 입력으로 전체 분석 실행 |
| 공통 함수 전체 재실행 | Notebook과 모델 데이터, 모델, Threshold, 예측, 네 Test 지표가 일치 |

수정한 두 스크립트는 모두 종료 코드 0으로 끝났다. 같은 분석용 입력으로 공통 함수도 돌려 Notebook과 비교했는데 모델 데이터, 모델, Threshold, 예측, 네 지표 모두 차이가 없었다. 선택 모델은 Logistic Regression, Threshold는 0.45이며 Validation F1은 0.56, Test F1은 0.29였다.

### 업무·분석적 의미

이번에는 두 스크립트와 공통 함수 재실행에서 모델, Threshold, 예측, 네 Test 지표의 차이가 모두 0이었다. Notebook과 스크립트의 결과가 같으니 분석 과정을 나중에 다시 검토할 수 있다. 분석용 입력과 제외 목록까지 남겨 두어 어떤 주문으로 결과를 얻었는지도 설명이 가능하다.

### 한계와 추가 확인 사항

분석 대상의 날짜 오류는 0건이지만 원본의 56건을 임의로 고치지는 않았다. 자동 검증 11개는 정해 둔 규칙대로 코드가 실행됐는지를 본 것이다. 제외한 완료/취소 45건이 남은 203건과 성격이 다른지는 이 검증으로 알 수 없다. 또 11개 중 4개는 내가 Notebook에서 추가한 검사라서 공통 모듈의 자동 검증만 돌리면 항목 수가 달라진다.

---

## 13. LLM 검토 프롬프트 예시

```text
온라인 쇼핑몰 주문 취소 분류 코드를 검토해 주세요.

타깃 기준:
- completed=0
- cancelled=1
- refunded와 기타 상태는 제외

검토 항목:
1. order_status와 is_cancelled가 feature에서 제외되었는가?
2. 병합에 validate와 indicator가 사용되었는가?
3. train, validation, test가 분리되었는가?
4. 모델과 임계값은 validation에서 선택했는가?
5. test는 최종 평가에 한 번만 사용했는가?
6. Dummy 기준 모델보다 나은가?
7. accuracy, precision, recall, f1을 함께 확인했는가?
8. 예측 결과에 개인정보가 포함되지 않았는가?
9. 모델 결과를 취소 원인으로 단정하지 않았는가?

필수 수정과 권장 개선을 구분해 설명해 주세요.
```


In [23]:
# LLM 검토 의견의 확인과 채택·수정·보류 기록
llm_review=pd.DataFrame([
 {'proposal':'병합의 validate·indicator 사용 여부를 코드에서 확인','evidence':f'병합 검증 2건 모두 행 수 유지, 미매칭 {int(merge_checks["unmatched_count"].sum())}건','decision':'채택','reason':'요약만으로는 LLM이 확인하지 못해 병합 검증표로 직접 확인'},
 {'proposal':'Test Accuracy가 Dummy보다 낮다는 점을 한계로 명시','evidence':f'Test Accuracy {float(test_metrics["accuracy"].iloc[0]):.2f}, 모두 완료로 예측하면 {float((y_test==0).mean()):.2f}','decision':'채택','reason':'Final Test 해석과 최종 판단에 반영'},
 {'proposal':'record_id가 order_id와 바로 연결되는지 점검','evidence':f'공개 {len(prediction_result_public)}행; 금지 컬럼 {len(privacy_forbidden.intersection(prediction_result_public.columns))}개','decision':'채택','reason':'record_id는 Test 순번이고 원래 행과의 대응은 내부 파일에만 있음'},
 {'proposal':'예측 결과를 취소 원인처럼 쓰지 않도록 문구 점검','evidence':'답안의 예측과 인과 구분 항목','decision':'채택','reason':'원인과 개입 효과는 확인하지 못했다고 작성'},
 {'proposal':'Test 성능 하락 이유(소표본, 제외로 인한 분포 차이, 과적합, 과잉 양성 예측)','evidence':f'Validation·Test 각 {len(y_test)}건, 취소 {int(y_test.sum())}건','decision':'수정','reason':'추가 분석을 하지 않아 가능한 이유로만 적고 단정하지 않음'},
 {'proposal':'FP를 먼저 줄여야 한다는 의견','evidence':f'Test FP {int(confusion_df.iloc[0,1])}건, FN {int(confusion_df.iloc[1,0])}건','decision':'수정','reason':'FP 부담은 인정하지만 실제 비용 자료가 없어 우선순위는 확정하지 않음'},
 {'proposal':'고객 단위 분할, 과거 집계 피처, 비용 기반 임계값·확률 보정','evidence':'분할 사이 고객 중복 확인','decision':'보류','reason':'이번 실습 범위를 넘어 다음 개선 과제로 남김'}])
display(llm_review);llm_review.to_csv(REPORT_DIR/'ch10_llm_review.csv',index=False,encoding='utf-8-sig')
summary_path=REPORT_DIR/'ch10_classification_summary.md'
summary_addition="\n\n## 최종 판단과 분석 범위\n\n현재 사용 보류. Logistic Regression, Threshold=0.45.\n203건(완료149/취소54), Train121/Validation41/Test41. Validation F1=0.562500, Test F1=0.285714.\nTest TN=11, FP=19, FN=6, TP=5. 실제 비용·인과·개입 효과는 미확인. 공개 예측41행, 최소6컬럼, 원본 식별자0개.\n분석 계약10개 및 입력 격리 계약1개 PASS, 원본 시간 오류56건은 격리 기록으로 보존. 수정한 스크립트 실행 성공.\nTest는 사전 선택 고정 후 최초 평가와 스크립트/공통 함수 재현으로 구분했다. 재현 결과로 선택을 바꾸지 않았다.\nrandom split의 고객 중복, 시간 일반화, 과거 Feature 가용성은 미검증.\n다음 우선순위: 원천 날짜 정정·분석 계약 통일 / 시간·고객별 외부 검증 / 실제 오류 비용·공정성·개입 효과 검증.\n검증된 Chapter10 입력의 전체 Validation PASS 및 수정한 스크립트 재현 완료. 원본 날짜 정정과 실제 운영 검증은 별도 미완료다.\n"
summary_path.write_text(summary_path.read_text(encoding='utf-8-sig')+summary_addition,encoding='utf-8')
print('검토 기록과 최종 판단 저장 완료')


,proposal,evidence,decision,reason
0,병합의 validate·indicator 사용 여부를 코드에서 확인,"병합 검증 2건 모두 행 수 유지, 미매칭 0건",채택,요약만으로는 LLM이 확인하지 못해 병합 검증표로 직접 확인
1,Test Accuracy가 Dummy보다 낮다는 점을 한계로 명시,"Test Accuracy 0.39, 모두 완료로 예측하면 0.73",채택,Final Test 해석과 최종 판단에 반영
2,record_id가 order_id와 바로 연결되는지 점검,공개 41행; 금지 컬럼 0개,채택,record_id는 Test 순번이고 원래 행과의 대응은 내부 파일에만 있음
3,예측 결과를 취소 원인처럼 쓰지 않도록 문구 점검,답안의 예측과 인과 구분 항목,채택,원인과 개입 효과는 확인하지 못했다고 작성
4,"Test 성능 하락 이유(소표본, 제외로 인한 분포 차이, 과적합, 과잉 양성 예측)","Validation·Test 각 41건, 취소 11건",수정,추가 분석을 하지 않아 가능한 이유로만 적고 단정하지 않음
5,FP를 먼저 줄여야 한다는 의견,"Test FP 19건, FN 6건",수정,FP 부담은 인정하지만 실제 비용 자료가 없어 우선순위는 확정하지 않음
6,"고객 단위 분할, 과거 집계 피처, 비용 기반 임계값·확률 보정",분할 사이 고객 중복 확인,보류,이번 실습 범위를 넘어 다음 개선 과제로 남김


검토 기록과 최종 판단 저장 완료


## 13. 최종 사용 판단

- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 사용 보류

### 판단 근거

1. **클래스 비율과 기준 모델 비교:** 분석한 203건은 완료 149건(73.40%), 취소 54건(26.60%)이었다. Validation에서 Dummy의 Precision, Recall, F1은 모두 0이었고 Logistic Regression은 각각 0.43, 0.82, 0.56이었다. 취소를 찾는 성능은 좋아졌지만 Accuracy는 0.73→0.66로 7.32%p 낮아졌다.
2. **Threshold와 최종 성능:** Validation에서 Threshold 0.45를 정했지만 0.50과 비교한 예측 차이는 0건이었다. Test의 Precision은 0.21, Recall은 0.45, F1은 0.29로 Validation보다 각각 0.22, 0.36, 0.28 낮았다. FP는 19건, FN은 6건이었으며 실제 오류 비용은 알 수 없었다.
3. **검증과 남은 문제:** 공개 결과는 41행 6컬럼이며 원본 식별자는 0개였다. 분석 검사 10개와 날짜 오류 주문의 제외 확인 1개가 모두 통과했고 스크립트와 Notebook의 결과도 같았다. 다만 원본 날짜 오류 56건(완료/취소 45건)은 날짜를 고치지 않고 제외한 상태다. 시간순 평가, 고객 중복 영향, 공정성, 실제 대응 효과도 확인하지 못했다.

### 최종 판단에 사용한 결과

| 확인한 내용 | 결과와 판단에 반영한 점 |
| --- | --- |
| 클래스 비율 | 완료 149건(73.40%), 취소 54건(26.60%)으로 불균형이 있었다. |
| Dummy 대비 개선 | Validation 취소 탐지는 0→9건(+9), F1은 0→0.56이었다. 완료 오탐지는 0→12건(+12)으로 늘었다. |
| Precision/Recall/F1 | Validation은 0.43/0.82/0.56, Test는 0.21/0.45/0.29였다. |
| FP/FN 비용 가정 | 취소 누락이 추가 확인보다 부담스러울 수 있다고 가정했다. 실제 건당 비용과 개입 효과가 없어 비용 차이는 확정하지 않았다. |
| Threshold 선택 이유 | Validation 최대 F1인 0.45와 0.50의 예측은 같았다. 공통 함수가 반환한 0.45를 고정했다. 0.60보다 FN은 5→2건(-3), FP는 6→12건(+6)이었다. |
| Final Test | Accuracy 0.39, TN 11, FP 19, FN 6, TP 5였다. 성능 하락을 반영해 새로운 주문에도 잘 맞는다고 판단하지 않았다. |
| random split의 한계 | 시간 순서를 반영하지 않았고 고객도 분할 사이에 겹쳤다. 실제 사용 전에는 시간순/out-of-time 평가가 필요하다. |
| 공개 결과 | 41행 6컬럼이며 source_index, order_id, customer_id, product_id와 불필요한 개인정보는 공개 컬럼에 없었다. |

### 나의 해석과 판단

나는 현재 상태로는 실제 업무 사용을 보류하는 편이 맞다고 봤다. 코드는 끝까지 실행되고 결과도 재현됐지만 Test에서 취소를 충분히 찾지 못했고 완료 주문을 위험하다고 표시하는 경우도 많았다. 실제 비용조차 모르는 상황에서 자동 검사 PASS만으로 사용을 결정하기는 어렵다.

### LLM 활용과 확인한 내용

분석 결과 검토는 LLM(Gemini)에 맡겼다. 처음에는 Notebook의 검토 프롬프트 예시만 보냈는데 코드가 없어서 검토할 수 없다는 답이 돌아왔다. 그래서 원본 데이터 대신 타깃 기준, 입력 변수, 분할과 선택 방식, Validation/Test 집계 결과를 정리해 다시 보냈다. 받은 의견은 그대로 옮기지 않고 실제 출력과 맞는지 하나씩 대조했다.

| LLM 의견 | 실제로 확인한 내용 | 결정과 이유 |
| --- | --- | --- |
| 병합에 validate와 indicator를 썼는지 요약만으로는 확인 불가, 코드에서 확인 필요 | 병합 검증표에서 두 병합 모두 203→203건, 미매칭 0건. 날짜 확인 병합에 `validate`와 `indicator` 사용 | 채택: LLM이 볼 수 없던 부분이라 직접 확인. 이미 되어 있어 코드는 그대로 둠 |
| Test Accuracy 0.39가 Dummy(0.73)보다 낮으니 숨기지 말고 한계로 적을 것 | Test 41건 중 완료가 30건이라 모두 완료로 예측하면 0.73. 지적이 맞음 | 채택: Final Test 해석과 최종 판단에 반영 |
| record_id가 order_id와 바로 연결되는지 점검할 것 | record_id는 Test 순번, 공개 6컬럼에 원본 식별자 0개. 원래 행과의 대응은 내부 파일에만 있음 | 채택: 공개 결과는 그대로 두고 내부 파일은 공개하지 않음 |
| 해석 문구를 보지 못해 원인 단정 여부는 확인 불가, 인과 표현 점검 필요 | 예측과 인과 구분 항목에서 원인과 개입 효과는 말할 수 없다고 작성 | 채택: 원인처럼 읽히는 표현이 없는지 다시 확인 |
| Test 성능 하락 이유로 소표본, 주문 제외로 인한 분포 차이, 과적합, 과잉 양성 예측을 제시 | Validation/Test 각 41건, 취소 11건. 어느 이유가 맞는지는 추가 분석을 하지 않음 | 수정: 가능한 이유로만 적고 원인으로 단정하지 않음 |
| FP(19건)를 먼저 줄여야 한다는 의견 | Test에서 완료 30건 중 19건이 FP인 것은 맞음. FP/FN의 실제 비용 자료는 없음 | 수정: FP 부담이 크다는 점은 받아들였지만 어느 오류가 더 큰지는 비용을 알아야 정할 수 있다고 봄 |
| 고객 단위 분할, 과거 집계 피처 추가, 비용 기반 임계값과 확률 보정 | 분할 사이에 같은 고객이 겹치는 것은 확인함 | 보류: 이번 실습 범위를 넘어서 다음 개선 과제로 남김 |

LLM이 병합 코드와 해석 문구를 보지 못해 확인 불가로 답한 두 항목은 내가 직접 점검했다. Test Accuracy가 Dummy보다 낮다는 지적은 맞는 말이어서 Final Test 해석에 반영했다. FP를 먼저 줄여야 한다는 의견은 비용 자료가 없어 그대로 받아들이지 않았다. 결과와 최종 판단은 `ch10_classification_summary.md`에도 기록했다.

### 업무·분석적 의미

Test에서 위험하다고 표시한 24건 중 실제 취소는 5건뿐이었고 실제 취소 11건 중 6건은 놓쳤다. 취소를 많이 맞히는 것 못지않게 불필요한 확인 업무와 놓치는 위험을 줄이는 일도 중요하다. 지금은 이 둘을 업무 비용과 연결할 자료가 부족해 자동 조치나 운영 판단에 쓰는 것은 보류했다.

### 현재 모델의 가장 큰 위험

이번 데이터는 일부 주문을 제외한 데다 규모도 작다. 그런 데이터에서 나온 Validation 결과를 앞으로의 모든 주문에도 적용할 수 있다고 믿는 것이 가장 큰 위험이다. 이번에는 Test에서 Precision과 Recall이 함께 떨어져 그런 판단을 뒷받침하기 어려웠다.

### 다음 개선 우선순위

1. 원천 날짜 확인: 가입일과 주문일을 원래 거래 기록과 대조하고 수정할 근거가 있으면 날짜를 고친 뒤 데이터 수, 분포, 성능을 다시 비교하고 싶다. 현재 Notebook과 스크립트는 같은 제외 기준으로 실행되니 원천 날짜를 정정하면 분석용 입력과 검사 결과도 함께 갱신해야 한다.
2. 평가 방법 보완: 과거 자료로 학습하고 이후 기간으로 평가하는 시간순 검증과 별도 holdout 평가가 필요하다. 신규 고객이 대상이라면 고객 단위 분할도 검토할 만하다. 변수가 주문 당시 정보인지, 확률이 실제 취소 가능성에 잘 맞는지도 확인해야 한다. 기존 Test를 개선 방법 선택에 다시 쓰지는 않아야 한다.
3. 실제 비용과 대응 효과 확인: FP/FN 건당 비용, 확인에 드는 인력과 시간, 취소 손실, 개입 성공률을 알아본 뒤 새로운 Validation에서 Threshold를 검토해야 한다. 고객 집단별로 오류와 부담이 다른지도 확인할 필요가 있다.

### 예측과 인과 구분
현재 모델에서 확인한 것은 어떤 **예측 패턴**이며, 어떤 **원인 주장**은 아직 할 수 없는지 작성하세요.

이번에 본 예측 패턴은 이렇다. 주문 금액과 수량, 주문 시기, 고객 정보, 결제수단처럼 주문 생성 시점에 알 수 있는 정보를 넣은 Logistic Regression은 Validation에서 취소 11건 중 9건을 찾았지만 Test에서는 11건 중 5건만 찾았다. 이 정보들과 취소 사이에 어느 정도 관련이 있을 수는 있지만 그 관련이 안정적이라고 보기는 어려웠다. 어떤 변수가 예측에 크게 작용했는지는 따로 분석하지 않았다. 그래서 특정 성별, 지역, 결제수단, 금액이 취소를 일으켰다고 말할 수 없다. 위험 주문에 연락했을 때 취소가 줄어드는지도 이 결과로는 판단할 수 없다. 원인이나 대응 효과를 말하려면 추가 자료와 별도 검증이 있어야 한다.

---

## 14. 정리

이번 장에서는 완료 주문과 취소 주문을 명확하게 구분하고, 병합 검증, 데이터 누수 방지, train/validation/test 분리, 기준 모델 비교, validation 기반 모델·임계값 선택, 최종 test 평가를 수행했습니다.

다음 장에서는 LLM을 활용해 분석 질문과 프롬프트를 더 구체적으로 만드는 방법으로 이어집니다.


## 최종 체크

- [x] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [x] `refunded`와 기타 상태를 제외했습니다.
- [x] 예측 시점과 Feature Contract를 정의했습니다.
- [x] target, ID, 사후 정보를 feature에서 제외했습니다.
- [x] `line_total = quantity × unit_price`를 검증했습니다.
- [x] 주문 단위 특징과 병합 관계를 검증했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] Train / Validation / Test 역할을 구분했습니다.
- [x] 모델을 Validation에서 선택했습니다.
- [x] Threshold를 Validation에서 선택했습니다.
- [x] Final Test 전에 선택을 고정했습니다.
- [x] Precision / Recall / F1을 해석했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Public prediction에서 식별자를 제거했습니다.
- [x] Validation Evidence를 확인했습니다.
- [x] random split의 한계를 기록했습니다.
- [x] 예측 패턴을 원인처럼 단정하지 않았습니다.
- [x] 최종 Notebook URL을 제출합니다.

### 추가 확인

- [x] 날짜 오류 주문의 제외 사유와 전후 건수를 기록하고 원본 날짜를 보존했다.
- [x] 전처리를 Pipeline 안에서 Train으로만 학습했다.
- [x] 필수 Evidence12개와 최종 검증11개 PASS를 확인했다.
- [x] 두 스크립트(로컬 수정본)를 실행하고 Notebook과 결과가 같음을 확인했다.
- [x] 제출용 chapter10.ipynb와 상대 경로 images 폴더를 구성했다.
- [x] 제출 정보를 입력하고 개인 GitHub 저장소에 업로드한다.
- [x] GitHub에서 Notebook의 답안·출력·표·그림이 정상적으로 보이는지 확인한다.

제출 정보와 GitHub 확인 항목은 직접 작성·확인한 뒤 체크한다. 현재 사용 보류는 모델의 업무 적용에 대한 판단이며, 실습 실행·분석을 완료하지 못했다는 뜻으로 사용하지 않았다.